# KVO_08 — Satellite Musanga fractional canopy prediction

**Integrated workflow (October 2026):** acquisition-year, uncorrected Sentinel-2 (annual and quarterly) and Google annual satellite embeddings → canonical native S2 10 m grids → frozen full-UAV U-Net probability targets → aligned feature/target tables → identical held-out-site validation for RAW, seasonal S2, embeddings and fusion. BRDF is deferred.

**Order:** 00 setup; 01 UAV years; 02 S2 grid registry; 03 UAV reference targets; **04 all satellite feature construction**; 05 import and align exports; 06 QA/common sample; 07 compare models; 08 multiscale 10–500 m; 09 diagnostics and BRDF future.

Existing validated RAW exports and UAV reference rasters are reused, not recomputed. Additional feature exports are opt-in and staged with unique site+family folder names before importing into the project. No user data is deleted or overwritten. Targets remain U-Net-derived proxies rather than independent field-measured fractional cover.

## 00 — Project paths and data contracts

In [ ]:
# BLOCK 00 — canonical site paths; CPU-only, read-only inspection
from pathlib import Path
import json, math, shutil
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from IPython.display import display
from google.colab import drive

MY = Path("/content/drive/MyDrive")
if not MY.exists(): drive.mount("/content/drive")
ROOT = MY / "Colab Notebooks" / "Musanga"
GEO = ROOT / "data" / "Geo_Ref"
KVO04 = ROOT / "KVO_work" / "KVO_04_final_multisite"
MAPS = KVO04 / "probability_maps"
OUT = ROOT / "KVO_work" / "KVO_08_satellite_fraction"
VALIDATION = OUT / "validation"
VALIDATION.mkdir(parents=True, exist_ok=True)

RGB = {
    "MOS_1_2": GEO / "MOS_1_2_kvo.tif",
    "Yoko": GEO / "Yoko_kvo_georef.tif",
    "Yangambi": GEO / "Yangambi_georef.tif",
    "Baego": GEO / "Baego_georef.tif",
    "Luki": GEO / "Luki.tif",
}
CURRENT = {site: MAPS / f"{site}_musanga_probability_uint16.tif" for site in RGB}
BACKUP = {
    site: MAPS / f"{site}_musanga_probability_uint16.pre_vector_roi_mask.tif"
    for site in RGB if site != "Luki"
}
FULL_LUKI = MAPS / "Luki_musanga_probability_uint16_FULL.tif"

NODATA = 65535
SCALE = 10000
SITES = list(RGB)
print("Validation outputs:", VALIDATION)
print("Sites:", ", ".join(SITES))
print("Luki FULL map present:", FULL_LUKI.exists())
print("No inference, overwrites, reprojecting, or downloads in Blocks 00–02.")

## 01 — UAV acquisition-year register

In [ ]:
# BLOCK 08 — Five-site acquisition years (provenance, CPU-only).
import pandas as pd
import json
UAV_ACQUISITION_YEAR = {
    "MOS_1_2": 2024,
    "Yoko": 2020,
    "Baego": 2023,
    "Yangambi": 2024,
    "Luki": 2023,
}
UAV_YEAR_SOURCE = {
    "MOS_1_2": "User acquisition table (Mos)",
    "Yoko": "User acquisition table",
    "Baego": "User acquisition table",
    "Yangambi": "User confirmation, 2026-10-11",
    "Luki": "User confirmation, 2026-10-11; supported by Plumacker et al. 2026, DOI:10.1002/ece3.74260",
}
UAV_DATE_NOTE = {
    "Luki": "Confirmed year 2023; exact date within year not verified (paper documents May and November dates)"
}
assert set(UAV_ACQUISITION_YEAR)==set(SITES)
years08=pd.DataFrame([
    dict(analysis_site=site,acquisition_year=UAV_ACQUISITION_YEAR[site],
         year_confirmed=True,source=UAV_YEAR_SOURCE[site],
         date_note=UAV_DATE_NOTE.get(site,""))
    for site in SITES
])
years08.to_csv(VALIDATION/"08_acquisition_years.csv",index=False)
display(years08)
print("All five annual acquisition years documented. Exact Luki acquisition DATE remains unresolved.")


## 02 — Native Sentinel-2 grid registry

Previously stored `validation/09_s2_canonical_10m_grids.json` is authoritative; rerun this stage only when rebuilding the grid registry.

In [ ]:
# BLOCK 02 — Year-matched uncorrected Sentinel-2 intake and canonical 10m grids
# CPU / Earth Engine; read-only metadata; no exports.
# Dependencies: run Blocks 00 and 08. Block 07 (2024 embedding probe) is NOT required.
import json, math, subprocess, sys
from pathlib import Path
import numpy as np, pandas as pd, rasterio
from rasterio.warp import transform_bounds
from IPython.display import display
try:
    import ee
except ImportError:
    subprocess.check_call([sys.executable,"-m","pip","-q","install","earthengine-api"])
    import ee
PROJECT09="musanga-511110"
try:
    ee.Initialize(project=PROJECT09)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT09)
assert "UAV_ACQUISITION_YEAR" in globals(), "Run Block 08 before Block 09."
assert set(UAV_ACQUISITION_YEAR)==set(SITES)
COLLECTION09="COPERNICUS/S2_SR_HARMONIZED"
S2_COLLECTION09=ee.ImageCollection(COLLECTION09)
REGISTRY09={}
REPORT09=[]
for site in SITES:
    year=int(UAV_ACQUISITION_YEAR[site])
    start=f"{year}-01-01";end=f"{year+1}-01-01"
    with rasterio.open(RGB[site]) as uav:
        assert uav.crs is not None and uav.crs.is_projected
        target_crs=str(uav.crs)
        bounds=[float(v) for v in uav.bounds]
        ll=transform_bounds(uav.crs,"EPSG:4326",*uav.bounds,densify_pts=21)
    region=ee.Geometry.Rectangle(list(ll),proj="EPSG:4326",geodesic=False)
    col=S2_COLLECTION09.filterDate(start,end).filterBounds(region)
    num=int(col.size().getInfo())
    if num==0:raise RuntimeError(f"{site}: no Sentinel-2 SR images in {year}")
    tiles=sorted(set(x for x in col.aggregate_array("MGRS_TILE").getInfo() if x))
    # Pick a native Sentinel-2 scene in the *same UTM zone* as the UAV.
    # Yangambi straddles S2 zones 34/35: never inherit the first zone-34 scene.
    zone=target_crs.split(":")[-1][-2:]
    preferred=[t for t in tiles if t.startswith(zone)]
    matched=[]
    for t in preferred:
        im=ee.Image(col.filter(ee.Filter.eq("MGRS_TILE",t)).first())
        p=im.select("B2").projection().getInfo()
        if p.get("crs")==target_crs:matched.append((t,p))
    if not matched:
        raise RuntimeError(
            f"{site}: no native S2 B2 tile on UAV CRS {target_crs}. "
            f"Available MGRS tiles: {tiles}. Do not silently choose a different UTM CRS.")
    chosen_tile,proj=matched[0]
    trans=proj["transform"]
    if not(len(trans)==6 and abs(trans[0]-10)<1e-8 and
           abs(trans[4]+10)<1e-8 and abs(trans[1])<1e-8 and abs(trans[3])<1e-8):
        raise RuntimeError(f"{site}: unexpected native S2 B2 grid: {proj}")
    # A canonical 10m window, snapped to exact S2 pixel origin, covering UAV bounds.
    ox,oy=float(trans[2]),float(trans[5])
    x0=ox+math.floor((bounds[0]-ox)/10)*10
    y0=oy-math.floor((oy-bounds[3])/10)*10
    ncols=math.ceil((bounds[2]-x0)/10)
    nrows=math.ceil((y0-bounds[1])/10)
    affine=[10,0,x0,0,-10,y0]
    # Relative integer scene-pixel shift, never a resampled arbitrary origin.
    assert np.allclose((x0-ox)/10,round((x0-ox)/10),atol=1e-5)
    assert np.allclose((y0-oy)/10,round((y0-oy)/10),atol=1e-5)
    registry=dict(site=site,uav_year=year,source_collection=COLLECTION09,
                  uav_crs=target_crs,s2_crs=proj["crs"],
                  s2_native_tile=chosen_tile,all_s2_tiles=tiles,
                  native_scene_transform=trans,
                  target_transform=affine,width=ncols,height=nrows,
                  uav_bbox_m=bounds,uav_bbox_lonlat=list(ll),
                  s2_granules_year=num,
                  note="S2 annual data only, NO BRDF adjustment; exact S2 B2 native grid")
    REGISTRY09[site]=registry
    REPORT09.append({k:registry[k] for k in (
        "site","uav_year","uav_crs","s2_native_tile","width","height","s2_granules_year")}|{
            "all_tiles":",".join(tiles),
            "target_x0":x0,"target_y0":y0})
    print(site, year, target_crs, "S2 scenes:",num,
          "tiles:",",".join(tiles),"selected:",chosen_tile,
          "grid:",ncols,"x",nrows)
pd.DataFrame(REPORT09).to_csv(VALIDATION/"09_s2_year_and_grid_registry.csv",index=False)
(VALIDATION/"09_s2_canonical_10m_grids.json").write_text(
    json.dumps(REGISTRY09,indent=2))
display(pd.DataFrame(REPORT09))
print("Success: native year-specific Sentinel-2 grid registry saved.")
print("No exports or BRDF processing performed.")


## 03 — UAV target rasters on the 10 m grid

Completed five-site outputs are reused automatically. Run only when target rasters are missing.

In [ ]:
# BLOCK 03 — Full-resolution UAV probability -> exact S2 10m analysis grid
# CPU ONLY. Run Block 00 first. Block 09 canonical grid JSON must already be on Drive.
# No Earth Engine, GPU, model training, or modifications to input rasters.
from pathlib import Path
import json,hashlib,math,shutil,time,os
import numpy as np,pandas as pd,rasterio
from affine import Affine
from rasterio.windows import Window,from_bounds,bounds as window_bounds
from rasterio.warp import reproject,Resampling,transform_bounds
from tqdm.auto import tqdm
from IPython.display import display

GRID_FILE11=VALIDATION/"09_s2_canonical_10m_grids.json"
assert GRID_FILE11.is_file(),(
    f"Missing {GRID_FILE11}. Run Block 09 once to save exact S2 grids.")
GRIDS11=json.loads(GRID_FILE11.read_text())
assert set(SITES)==set(GRIDS11),"Canonical grid register does not contain all five sites."
FULL11={s:MAPS/f"{s}_musanga_probability_uint16_FULL.tif" for s in SITES}
DESTDIR11=OUT/"uav_targets_10m"
DESTDIR11.mkdir(parents=True,exist_ok=True)
RUN_SITES11=list(SITES) # set e.g. ["Luki"] to run one site
TILE11=32             # 32x32 target cells => bounded UAV reads, safe RAM
COVERAGE_MIN11=0.90   # gate reference target metrics below 90% observed UAV coverage
THRESHOLD11=0.50      # frozen pixel-level Musanga classification threshold
NODATA11=-9999.0
BANDS11=[
  "musanga_probability_mean",
  "musanga_cover_fraction_p05",
  "uav_valid_fraction",
  "musanga_probability_sd",
]
PIPELINE11="UAV_S2_10M_weighted_v1_mean_fraction_coverage_spatial_sd"
assert 0<COVERAGE_MIN11<=1 and 0<=THRESHOLD11<=1
assert set(RUN_SITES11).issubset(SITES)

def project_info11(site):
    g=GRIDS11[site]
    trans=Affine(*g["target_transform"])
    if not(np.isclose(trans.a,10) and np.isclose(trans.e,-10) and
           np.isclose(trans.b,0) and np.isclose(trans.d,0)):
        raise ValueError((site,"unexpected 10m S2 transform",g["target_transform"]))
    return g,trans

def run_target11(site):
    g,dst_transform=project_info11(site)
    source=FULL11[site]
    assert source.is_file(),f"{site}: full UAV probability map missing: {source}"
    assert RGB[site].is_file(),f"{site}: original UAV mosaic missing"
    with rasterio.open(source) as src:
        assert src.dtypes[0]=="uint16" and src.nodata==65535,(site,"invalid source encoding")
        assert str(src.crs)==g["uav_crs"],(site,src.crs,g["uav_crs"])
        assert src.tags().get("COMPLETE")=="YES" and src.tags().get("COVERAGE")=="FULL_VALID_RGB",(
            site,"full UAV map lacks production-complete metadata")
        assert src.tags().get("MUSANGA_SCALE")=="10000"
        src_signature=src.tags().get("SPEC_FINGERPRINT","")
        assert src_signature,(site,"no frozen-model provenance fingerprint")
        width=int(g["width"]);height=int(g["height"])
        crs=g["uav_crs"]
    fingerprint_input=dict(
        site=site,source_signature=src_signature,
        target_transform=list(g["target_transform"]),target_crs=crs,
        target_width=width,target_height=height,
        pipeline=PIPELINE11,threshold=THRESHOLD11,
        min_coverage=COVERAGE_MIN11
    )
    fingerprint=hashlib.sha256(json.dumps(fingerprint_input,sort_keys=True).encode()).hexdigest()[:20]
    final=DESTDIR11/f"{site}_musanga_uav_target_S2grid_10m.tif"
    if final.exists():
        with rasterio.open(final) as old:
            ok=(old.count==4 and old.width==width and old.height==height and
                old.crs==rasterio.crs.CRS.from_string(crs) and
                old.transform==dst_transform and old.nodata==NODATA11 and
                old.tags().get("FINGERPRINT")==fingerprint and
                old.tags().get("COMPLETE")=="YES")
        if not ok:
            raise RuntimeError(f"Existing target incompatible, inspect before re-running: {final}")
        print("Verified existing aligned UAV target, SKIP:",site)
        return dict(site=site,status="EXISTS",file=str(final),width=width,height=height)
    print(f"\n{site} | {width} x {height} | {crs} | 10 m S2 pixel grid")
    tmp=Path(f"/content/{site}_musanga_target_S2grid_10m_partial.tif")
    tmp.unlink(missing_ok=True)
    profile=dict(driver="GTiff",height=height,width=width,count=4,dtype="float32",
                 crs=crs,transform=dst_transform,nodata=NODATA11,
                 compress="deflate",predictor=3,tiled=True,
                 blockxsize=256,blockysize=256)
    # Use a local staged output then publish only once all windows complete.
    start=time.time();accepted=0;partial=0;observed=0;total=0
    try:
        with rasterio.open(source) as src, rasterio.open(tmp,"w+",**profile) as dst:
            for k,name in enumerate(BANDS11,1):
                dst.set_band_description(k,name)
            # Fill every output pixel with explicit NoData. This file is small (<10MB/site).
            for rr in range(0,height,256):
                for cc in range(0,width,256):
                    hh=min(256,height-rr);ww=min(256,width-cc)
                    initial=np.full((4,hh,ww),NODATA11,np.float32)
                    initial[2]=0.0   # no observed UAV pixels = zero valid-coverage fraction
                    dst.write(initial,window=Window(cc,rr,ww,hh))
            nrows=(height+TILE11-1)//TILE11
            for r0 in tqdm(range(0,height,TILE11),total=nrows,desc=f"{site} UAV->S2 10m rows"):
                for c0 in range(0,width,TILE11):
                    hh=min(TILE11,height-r0);ww=min(TILE11,width-c0)
                    target_win=Window(c0,r0,ww,hh)
                    tb=window_bounds(target_win,dst_transform)
                    # Reprojection footprint transformation also handles different CRS.
                    sb=transform_bounds(crs,src.crs,*tb,densify_pts=9)
                    sw=from_bounds(*sb,transform=src.transform)
                    margin=3
                    # Keep the FULL destination cell footprint in source coordinates,
                    # including pixels beyond the UAV raster boundary.
                    # Boundless 65535 pixels map to validity=0, allowing correct
                    # fractional edge-cell coverage rather than edge normalization.
                    sr0=int(math.floor(sw.row_off))-margin
                    sc0=int(math.floor(sw.col_off))-margin
                    sr1=int(math.ceil(sw.row_off+sw.height))+margin
                    sc1=int(math.ceil(sw.col_off+sw.width))+margin
                    if sr1<=0 or sc1<=0 or sr0>=src.height or sc0>=src.width:
                        continue
                    src_win=Window(sc0,sr0,sc1-sc0,sr1-sr0)
                    pix=src.read(1,window=src_win,boundless=True,fill_value=65535)
                    if not np.any(pix!=65535):
                        continue
                    local_transform=src.window_transform(src_win)
                    valid=pix!=65535
                    # All source-to-grid resampling is pixel-area averaging.
                    # For invalid pixels use zero numerator and zero validity.
                    # Divide by averaged validity rather than assign missing as absence.
                    work=np.zeros(pix.shape,dtype=np.float32)
                    work[valid]=pix[valid].astype(np.float32)/10000.0
                    def aggregate_mean11(array):
                        target=np.zeros((hh,ww),dtype=np.float32)
                        reproject(source=array,destination=target,
                                  src_transform=local_transform,src_crs=src.crs,
                                  dst_transform=dst.window_transform(target_win),
                                  dst_crs=dst.crs,src_nodata=None,dst_nodata=0,
                                  resampling=Resampling.average,num_threads=2,
                                  init_dest_nodata=True)
                        return target
                    fraction=aggregate_mean11(valid.astype(np.float32))
                    numerator=aggregate_mean11(work)
                    # Second moment and fixed probability >=0.5 indicator.
                    np.square(work,out=work)
                    square_num=aggregate_mean11(work)
                    work[:]=0
                    work[valid]=(pix[valid]>=int(round(THRESHOLD11*10000))).astype(np.float32)
                    cover_num=aggregate_mean11(work)
                    observed+=int((fraction>0).sum())
                    eligible=fraction>=COVERAGE_MIN11-1e-5
                    accepted+=int(eligible.sum())
                    partial+=int(((fraction>0)&~eligible).sum())
                    out=np.full((4,hh,ww),NODATA11,dtype=np.float32)
                    # valid coverage stays 0..1 even for cells not eligible for targets
                    out[2]=np.clip(fraction,0,1)
                    safe=np.maximum(fraction,1e-9)
                    mu=np.clip(numerator/safe,0,1)
                    variance=np.maximum(square_num/safe-mu*mu,0)
                    out[0,eligible]=mu[eligible]
                    out[1,eligible]=np.clip(cover_num/safe,0,1)[eligible]
                    out[3,eligible]=np.sqrt(variance[eligible])
                    dst.write(out,window=target_win)
                    total+=1
            dst.update_tags(
                COMPLETE="YES",FINGERPRINT=fingerprint,SOURCE_PROBABILITY=str(source),
                SOURCE_SPEC_FINGERPRINT=src_signature,ANALYSIS_GRID="native Sentinel-2 B2 10m",
                UAV_VALID_MIN=str(COVERAGE_MIN11),MUSANGA_THRESHOLD=str(THRESHOLD11),
                RESAMPLING="area average: numerator / valid area, non-observed pixels excluded",
                PROB_SD="spatial within-S2-cell variation, not model prediction uncertainty")
        with rasterio.open(tmp) as chk:
            assert chk.crs==rasterio.crs.CRS.from_string(crs)
            assert chk.transform==dst_transform
            assert chk.width==width and chk.height==height and chk.count==4
            assert chk.tags().get("FINGERPRINT")==fingerprint
            assert chk.descriptions==tuple(BANDS11)
        assert not final.exists()
        shutil.copy2(tmp,final)
        assert final.stat().st_size==tmp.stat().st_size
        tmp.unlink(missing_ok=True)
    except Exception:
        print("Failed site:",site,"Temporary partial output may exist:",tmp)
        raise
    qa=dict(site=site,status="COMPLETE",file=str(final),
            crs=crs,width=width,height=height,valid_reference_cells=accepted,
            partially_observed_cells_below_coverage_threshold=partial,
            sampled_or_partially_observed_cells=observed,
            min_coverage=COVERAGE_MIN11,threshold=THRESHOLD11,
            fingerprint=fingerprint,source_model_fingerprint=src_signature,
            runtime_minutes=round((time.time()-start)/60,2))
    (VALIDATION/f"11_{site}_target_10m_QA.json").write_text(json.dumps(qa,indent=2))
    print("COMPLETED",site,"accepted 10m cells:",accepted,
          "output MB:",round(final.stat().st_size/1e6,1))
    return qa

results11=[run_target11(s) for s in RUN_SITES11]
summary11=pd.DataFrame(results11)
summary11.to_csv(VALIDATION/"11_uav_target_10m_summary.csv",index=False)
display(summary11)
print("All requested 10m UAV targets complete; native UAV rasters unchanged.")


In [ ]:
# BLOCK 03L — Correct Luki UAV probability georeferencing (3 paired GCPs)
# IMPORTANT: columns 1-2 assumed ORIGINAL XY, columns 3-4 CORRECTED XY,
# in Luki's existing UTM CRS (EPSG:32733). Check this direction before execution.
# Run Blocks 00 and 03 first. No inference, no Earth Engine and no other site modified.
# Preserves full-resolution uint16 pixel values. Fits a 2D affine (including rotation/shear).
import json,shutil,hashlib
from pathlib import Path
import numpy as np,rasterio
from affine import Affine

LUKI_OLD_XY03L=np.array([
    [289355.331250,9378298.115416],
    [289887.047604,9377653.025741],
    [289176.719321,9377532.373705]],dtype=np.float64)
LUKI_NEW_XY03L=np.array([
    [289360.828531,9378303.451012],
    [289890.928037,9377658.231988],
    [289180.994707,9377539.349260]],dtype=np.float64)
# Switch off if old/new columns are reversed. Default is safe: do nothing until enabled.
APPLY_LUKI_CORRECTION03L=False

def run_luki_relocation03L():
    old=LUKI_OLD_XY03L; new=LUKI_NEW_XY03L
    # Centering makes the affine fit numerically stable at ~9.38 million northing.
    origin=old.mean(axis=0)
    local=np.column_stack((old-origin, np.ones(len(old))))
    coef=np.linalg.solve(local,new)
    m=coef[:2,:].T
    b=coef[2,:]-m@origin
    geo=Affine(m[0,0],m[0,1],b[0],m[1,0],m[1,1],b[1])
    residual=np.column_stack((old,np.ones(3)))@np.vstack((m.T,b))-new
    change=new-old
    print("Luki point shifts in metres (E, N):",np.round(change,3).tolist())
    print("Mean shift (E, N):",np.round(change.mean(axis=0),3).tolist())
    print("Affine transform (original projected XY -> corrected projected XY):",geo)
    print("GCP residuals (m; exactly three points fit exactly):",np.round(residual,6).tolist())
    svals=np.linalg.svd(m,compute_uv=False)
    if not (np.all((svals>0.99)&(svals<1.01)) and np.linalg.det(m)>0):
        raise RuntimeError("Unexpected scale/shear; inspect GCP order and coordinate system.")
    original=FULL11["Luki"]
    corrected=original.with_name(original.stem+"_GEOREF_20261012.tif")
    archive=DESTDIR11/"Luki_musanga_uav_target_S2grid_10m_BEFORE_GEOREF_20261012.tif"
    final=DESTDIR11/"Luki_musanga_uav_target_S2grid_10m.tif"
    g,grid=project_info11("Luki")
    with rasterio.open(original) as src:
        assert src.crs.to_epsg()==32733,(src.crs,"Expected Luki UTM 33S")
        assert src.count==1 and src.nodata==65535 and src.dtypes[0]=="uint16"
        original_transform=src.transform
        corrected_transform=geo*original_transform
        original_tags=src.tags()
    if corrected.exists():
        with rasterio.open(corrected) as check:
            if not check.transform.almost_equals(corrected_transform):
                raise RuntimeError("Existing corrected raster has a different transform; refusing overwrite.")
            assert check.tags().get("GEOREF_CORRECTION")=="20261012_affine_3_pairs"
        print("Verified existing corrected probability raster:",corrected)
    else:
        # Copy original byte-for-byte first; update ONLY transform metadata on copy.
        staging=corrected.with_suffix(".incoming.tif")
        if staging.exists():staging.unlink()
        shutil.copy2(original,staging)
        with rasterio.open(staging,"r+") as out:
            out.transform=corrected_transform
            out.update_tags(GEOREF_CORRECTION="20261012_affine_3_pairs",
                GEOREF_ORIGINAL=str(original),GEOREF_AFFINE=json.dumps(list(geo)[:6]),
                GEOREF_GCP_OLD=json.dumps(old.tolist()),
                GEOREF_GCP_NEW=json.dumps(new.tolist()))
        with rasterio.open(staging) as check, rasterio.open(original) as raw:
            assert check.transform.almost_equals(corrected_transform)
            assert check.count==raw.count and check.shape==raw.shape
            assert check.tags().get("SPEC_FINGERPRINT")==raw.tags().get("SPEC_FINGERPRINT")
            assert np.array_equal(check.read(1,window=rasterio.windows.Window(0,0,min(128,raw.width),min(128,raw.height))),
                                  raw.read(1,window=rasterio.windows.Window(0,0,min(128,raw.width),min(128,raw.height))))
        staging.rename(corrected)
        print("Saved corrected probability raster (pixel values unchanged):",corrected)
    # Archive old 10m reference rather than delete. Re-run only Luki aggregation.
    if final.exists() and not archive.exists():
        shutil.copy2(final,archive)
        print("Archived previous Luki 10m UAV target:",archive)
    elif final.exists() and archive.exists():
        with rasterio.open(final) as check:
            if check.tags().get("LUKI_GEOREF_CORRECTION")=="20261012_affine_3_pairs":
                print("Already rebuilt corrected Luki 10m target:",final)
                return
        raise RuntimeError("Both corrected-target destination and archive exist, but destination has no correction provenance. Inspect manually.")
    FULL11["Luki"]=corrected
    # The production target writer intentionally refuses overwrites.
    if final.exists():final.unlink() # only after preserving archived copy
    qa=run_target11("Luki")
    with rasterio.open(final,"r+") as ds:
        ds.update_tags(LUKI_GEOREF_CORRECTION="20261012_affine_3_pairs",
                       LUKI_GEOREF_SOURCE=str(corrected))
    provenance=dict(old_points=old.tolist(),new_points=new.tolist(),
                    affine=list(geo)[:6],mean_shift_m=change.mean(axis=0).tolist(),
                    old_target_backup=str(archive),corrected_probability=str(corrected),
                    target_10m=str(final),qa=qa)
    (VALIDATION/"Luki_affine_georeference_correction_20261012.json").write_text(
        json.dumps(provenance,indent=2,default=str))
    print("DONE: regenerated Luki-only S2-aligned UAV targets. All other sites unchanged.")
if APPLY_LUKI_CORRECTION03L:
    run_luki_relocation03L()
else:
    print("Luki GCP transform staged. Verify original->corrected column order,")
    print("then set APPLY_LUKI_CORRECTION03L=True and run this cell.")


## 04 — Unified rich satellite feature extraction, saved directly into the Musanga project

The new **three-year S2 temporal stack has 168 predictors** (three years × four calendar quarters × ten reflectance bands and four spectral indices). The **three-year annual embedding stack has 192 predictors** (three years × 64 embeddings). Their **360-predictor fusion** matches the earlier NB06 reported feature *counts*, but the exact older temporal aggregation, feature definitions and validation should not be assumed identical without recovering that notebook.

Other available families remain RAW_ANNUAL (15), S2_QUARTERLY (44), EMB_ANNUAL (64), and their fusion (123). Years are centered on UAV acquisitions: Yoko 2019–2021; Baego and Luki 2022–2024; MOS and Yangambi 2023–2025.

**Direct project download:** install `geemap geedim` in Colab, then set `START_DIRECT04=True` and leave `START_EXPORTS04=False`. Block 04 retrieves tiled Earth Engine images and writes GeoTIFFs to `KVO_work/KVO_08_satellite_fraction/satellite_features/<family>/`, with exact 10 m grid, band-count, CRS and NoData verification. Already completed products are reused. Earth Engine Drive export folders are not used.

The three-year band prefixes are `Ym1` (preceding year), `Y0` (UAV year), and `Yp1` (following year), avoiding invalid Earth Engine `+` characters.\n\nThe existing RAW annual baseline, UAV targets and prior model results remain untouched. These three-year predictors use imagery after the UAV year for retrospective inference; this is not a causal real-time forecast.


In [ ]:
# BLOCK 04 — Single, coherent feature family definition and extraction stage
# Run 00-02 (or use the saved grid JSON). The RAW export exists; no re-export!
from pathlib import Path
import json,pandas as pd
import ee
GRIDS04=json.loads((VALIDATION/"09_s2_canonical_10m_grids.json").read_text())
assert set(GRIDS04)==set(SITES)
YEAR_OFFSET_TAG04={-1:"Ym1",0:"Y0",1:"Yp1"}
S2_BASE04=["B2","B3","B4","B8","B5","B6","B7","B8A","B11","B12"]
S2_INDICES04=["NDVI","NDMI","NBR","NDRE"]
EMB_NAMES04=[f"A{i:02d}" for i in range(64)]
FEATURES04={
 "RAW_ANNUAL":S2_BASE04+S2_INDICES04+["N_CLEAR_S2"],
 "S2_QUARTERLY":[f"Q{q}_{b}" for q in range(1,5) for b in S2_BASE04+["N_CLEAR"]],
 "EMB_ANNUAL":EMB_NAMES04,
 "S2_T3Y":[f"{YEAR_OFFSET_TAG04[d]}_Q{q}_{b}" for d in (-1,0,1) for q in range(1,5) for b in S2_BASE04+S2_INDICES04],
 "EMB_T3Y":[f"{YEAR_OFFSET_TAG04[d]}_{b}" for d in (-1,0,1) for b in EMB_NAMES04]
}
FEATURES04["FUSION"]=FEATURES04["RAW_ANNUAL"]+FEATURES04["S2_QUARTERLY"]+FEATURES04["EMB_ANNUAL"]
FEATURES04["FUSION_T3Y"]=FEATURES04["S2_T3Y"]+FEATURES04["EMB_T3Y"]
RAW_PROJECT04=OUT/"satellite_raw_s2_10m"
EXTRA_PROJECT04=OUT/"satellite_features"
EXTRA_PROJECT04.mkdir(parents=True,exist_ok=True)
SRC04={"s2":"COPERNICUS/S2_SR_HARMONIZED",
       "emb":"GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL"}
REJECT_SCL04=[0,1,3,7,8,9,10,11]
contract04=dict(features=FEATURES04,collections=SRC04,rejected_scl=REJECT_SCL04,
                year_by_site={s:int(GRIDS04[s]["uav_year"]) for s in SITES},
                crs_transform_source="09_s2_canonical_10m_grids.json",
                brdf=False,scale_s2=.0001)
(VALIDATION/"04_feature_contract.json").write_text(json.dumps(contract04,indent=2))
display(pd.DataFrame([dict(family=f,n_features=len(v)) for f,v in FEATURES04.items()]))
START_EXPORTS04=False
EXPORT_SITES04=list(SITES)
EXPORT_FAMILIES04=["S2_QUARTERLY","EMB_ANNUAL"] # optionally add "RAW_ANNUAL" ONLY for deliberate fresh regeneration
def s2_mask04(image):
    scl=image.select("SCL")
    valid=ee.Image(1)
    for cat in REJECT_SCL04:valid=valid.And(scl.neq(cat))
    valid=valid.And(image.select("B8A").mask()).And(image.select("B9").mask())
    ten=image.select(S2_BASE04[:4])
    twenty=image.select(S2_BASE04[4:]).resample("bilinear")
    return ee.Image.cat([ten,twenty]).updateMask(valid).copyProperties(image,["system:time_start"])
def ee_region04(site):
    return ee.Geometry.Rectangle(GRIDS04[site]["uav_bbox_lonlat"],
                                  proj="EPSG:4326",geodesic=False)
def create_feature_image04(site,family):
    year=int(GRIDS04[site]["uav_year"])
    geom=ee_region04(site)
    if family in ("EMB_ANNUAL","EMB_T3Y"):
        offsets=(0,) if family=="EMB_ANNUAL" else (-1,0,1)
        images=[]
        for offset in offsets:
            yr=year+offset
            col=(ee.ImageCollection(SRC04["emb"]).filterBounds(geom)
                 .filterDate(f"{yr}-01-01",f"{yr+1}-01-01"))
            count=int(col.size().getInfo())
            if not count:
                raise RuntimeError(f"{site} {yr}: annual embeddings unavailable")
            label=EMB_NAMES04 if family=="EMB_ANNUAL" else [
                f"{YEAR_OFFSET_TAG04[offset]}_{b}" for b in EMB_NAMES04]
            images.append(col.mosaic().select(EMB_NAMES04).rename(label))
        return ee.Image.cat(images).toFloat().clip(geom)
    if family in ("S2_QUARTERLY","S2_T3Y"):
        offsets=(0,) if family=="S2_QUARTERLY" else (-1,0,1)
        layers=[]
        for offset in offsets:
            yr=year+offset
            col=(ee.ImageCollection(SRC04["s2"]).filterBounds(geom)
                 .filterDate(f"{yr}-01-01",f"{yr+1}-01-01")).map(s2_mask04)
            for q in range(1,5):
                begin=f"{yr}-{3*q-2:02d}-01"
                end=f"{yr+1}-01-01" if q==4 else f"{yr}-{3*q+1:02d}-01"
                part=col.filterDate(begin,end)
                if int(part.size().getInfo())==0:
                    raise RuntimeError(f"{site} {yr} Q{q}: no Sentinel-2 imagery")
                med=part.median().multiply(.0001)
                if family=="S2_T3Y":
                    indices=ee.Image.cat([
                        med.normalizedDifference(["B8","B4"]).rename("NDVI"),
                        med.normalizedDifference(["B8","B11"]).rename("NDMI"),
                        med.normalizedDifference(["B8","B12"]).rename("NBR"),
                        med.normalizedDifference(["B8A","B5"]).rename("NDRE")])
                    names=[f"{YEAR_OFFSET_TAG04[offset]}_Q{q}_{b}" for b in S2_BASE04+S2_INDICES04]
                    layers.append(ee.Image.cat([med,indices]).rename(names))
                else:
                    layers.append(med.rename([f"Q{q}_{b}" for b in S2_BASE04]))
                    layers.append(part.select("B2").count().unmask(0).toFloat()
                                  .rename(f"Q{q}_N_CLEAR"))
        return ee.Image.cat(layers).toFloat().clip(geom)
    if family=="RAW_ANNUAL":
        col=(ee.ImageCollection(SRC04["s2"]).filterBounds(geom)
             .filterDate(f"{year}-01-01",f"{year+1}-01-01")).map(s2_mask04)
        med=col.median().multiply(.0001)
        ind=ee.Image.cat([
            med.normalizedDifference(["B8","B4"]).rename("NDVI"),
            med.normalizedDifference(["B8","B11"]).rename("NDMI"),
            med.normalizedDifference(["B8","B12"]).rename("NBR"),
            med.normalizedDifference(["B8A","B5"]).rename("NDRE")])
        return ee.Image.cat([med,ind,col.select("B2").count().unmask(0).rename("N_CLEAR_S2")]).toFloat().clip(geom)
    raise ValueError(f"Family {family} is virtual or unknown")


# Preferred mode: Colab tile download directly to the canonical Musanga project.
# Requires: !pip -q install geemap geedim  (once per Colab runtime).
# Never export the virtual FUSION or FUSION_T3Y stacks separately.
START_DIRECT04=False
DIRECT_FAMILIES04=["S2_T3Y","EMB_T3Y","S2_QUARTERLY","EMB_ANNUAL"]
DIRECT_SITES04=list(SITES)
def download_direct04(site,family):
    import os,shutil,rasterio,numpy as np
    from affine import Affine
    # geemap is often preinstalled in Colab, but its image-download
    # backend geedim is not. Check both before starting any EE work.
    import importlib.util,subprocess,sys
    missing=[name for name in ("geemap","geedim") if importlib.util.find_spec(name) is None]
    if missing:
        print("Installing missing Earth Engine download dependencies:",missing,flush=True)
        subprocess.check_call([sys.executable,"-m","pip","install","-q",*missing])
    import geemap,geedim
    print("   geemap/geedim available; proceeding",flush=True)
    g=GRIDS04[site]
    yr=int(g["uav_year"])
    outdir=EXTRA_PROJECT04/family
    outdir.mkdir(parents=True,exist_ok=True)
    final=outdir/f"{site}_{family}_{yr}_10m.tif"
    expected=Affine(*g["target_transform"])
    def check(path):
        with rasterio.open(path) as ds:
            assert ds.count==len(FEATURES04[family]),(site,family,"bands")
            assert (ds.height,ds.width)==(g["height"],g["width"]),(site,family,"shape")
            assert str(ds.crs)==g["uav_crs"],(site,family,"CRS")
            assert ds.transform.almost_equals(expected,precision=8),(site,family,"grid")
            assert ds.nodata==-9999,(site,family,"NoData")
    if final.exists():
        check(final)
        print("VERIFIED existing feature:",final)
        return
    temp=Path("/content")/f"nb08_{site}_{family}_{yr}.partial.tif"
    staging=final.with_suffix(".incoming")
    # Suppress ONLY repetitive benign warnings from temporary many-band EE TIFF
    # tiles. Exception tracebacks and warnings from other libraries remain visible.
    import logging,warnings
    logging.getLogger("rasterio._env").setLevel(logging.ERROR)
    logging.getLogger("urllib3.connectionpool").setLevel(logging.ERROR)
    warnings.filterwarnings("ignore",message=".*Couldn't find STAC entry.*",
                            category=RuntimeWarning,module=r"geedim\.image")
    warnings.filterwarnings("ignore",message=".*BaseImage.*deprecated.*",
                            category=FutureWarning,module=r"geemap\.common")
    warnings.filterwarnings("ignore",message=".*num_threads.*deprecated.*",
                            category=FutureWarning,module=r"geemap\.common")
    if staging.exists():
        raise RuntimeError(f"Interrupted Google Drive staging file: {staging}. Inspect before rerun.")
    def verify_payload(path):
        # NoData is often missing in geedim-created GeoTIFF metadata even
        # when Earth Engine image.unmask(-9999) supplied correct pixel values.
        with rasterio.open(path) as ds:
            assert ds.count==len(FEATURES04[family]),(site,family,"bands",ds.count)
            assert (ds.height,ds.width)==(g["height"],g["width"]),(site,family,"shape",ds.shape)
            assert str(ds.crs)==g["uav_crs"],(site,family,"CRS",ds.crs)
            assert ds.transform.almost_equals(expected,precision=8),(site,family,"grid",ds.transform)
            assert ds.dtypes[0]=="float32",(site,family,"dtype",ds.dtypes[0])
            # Read the whole payload to detect truncated/corrupt tiles,
            # not merely a TIFF header that looks correct.
            for band in range(1,ds.count+1):
                for _,window in ds.block_windows(band):
                    values=ds.read(band,window=window)
                    if not np.isfinite(values).all():
                        raise RuntimeError(f"{site}/{family}: NaN/Inf in band {band}")
    if temp.exists():
        print(f"   Found completed local scratch ({temp.stat().st_size/1048576:.1f} MiB); checking for safe reuse...",flush=True)
        verify_payload(temp)
        print("   Existing pixel data verified — no re-download needed.",flush=True)
    else:
        print(f"   Preparing Earth Engine {site} / {family} ({len(FEATURES04[family])} bands)...",flush=True)
        img=create_feature_image04(site,family).unmask(-9999,sameFootprint=False).toFloat()
        print("   Downloading Earth Engine tiles to Colab scratch...",flush=True)
        geemap.download_ee_image(
            img,filename=str(temp),crs=g["uav_crs"],
            crs_transform=g["target_transform"],shape=(g["height"],g["width"]),
            dtype="float32",resampling="near",max_tile_dim=256,
            num_threads=1,overwrite=False)
        print(f"   Local tiles assembled ({temp.stat().st_size/1048576:.1f} MiB); validating...",flush=True)
        verify_payload(temp)
    with rasterio.open(temp,"r+") as ds:
        # geedim's default float32 TIFF NoData can be -inf even when the
        # requested EE unmask sentinel is -9999. verify_payload() above read
        # EVERY pixel and already rejected NaN/Inf, so replacing the metadata
        # does not misinterpret any actual -inf pixels.
        old_nodata=ds.nodata
        if old_nodata is None or (np.isinf(old_nodata) and old_nodata<0):
            ds.nodata=-9999.0
            print(f"   Normalized GDAL NoData metadata {old_nodata} -> -9999 (all pixel values verified finite; unchanged).",flush=True)
        elif old_nodata!=-9999:
            raise RuntimeError(f"{site}/{family}: unexpected NoData={old_nodata}; refusing to publish.")
        for band,name in enumerate(FEATURES04[family],1):
            ds.set_band_description(band,name)
        ds.update_tags(FAMILY=family,ACQUISITION_YEAR=str(yr),
                       TEMPORAL="year-1/year/year+1" if family.endswith("T3Y") else "acquisition-year",
                       RAW_S2_NO_BRDF="NO" if (family.startswith("S2_BRDF") or family.startswith("S2_AMAP")) else "YES",
                       BRDF_METHOD="RossLi_Roy2017" if family.startswith("S2_BRDF") else ("Ploton_S2_AMAP_July2026" if family.startswith("S2_AMAP") else "none"))
    check(temp)
    print("   Copying verified raster into Musanga Google Drive folder...",flush=True)
    shutil.copyfile(temp,staging)
    check(staging)
    os.replace(staging,final)
    temp.unlink()
    print("SAVED to canonical Musanga project:",final)
if START_DIRECT04:
    try:
        ee.Initialize(project="musanga-511110")
    except Exception:
        ee.Authenticate()
        ee.Initialize(project="musanga-511110")
    import time
    total04=len(DIRECT_SITES04)*len(DIRECT_FAMILIES04)
    done04=0
    started04=time.monotonic()
    print(f"Starting DIRECT satellite extraction: {total04} site × family products",flush=True)
    for site in DIRECT_SITES04:
        for family in DIRECT_FAMILIES04:
            tick04=time.monotonic()
            print(f"\n[{done04+1}/{total04}] {site} — {family} | {len(FEATURES04[family])} bands | elapsed {(tick04-started04)/60:.1f} min",flush=True)
            download_direct04(site,family)
            done04+=1
            elapsed04=time.monotonic()-started04
            print(f"[{done04}/{total04}] COMPLETED {site}/{family} in {(time.monotonic()-tick04)/60:.1f} min | total {elapsed04/60:.1f} min",flush=True)
    print(f"ALL {done04} direct feature products finished in {(time.monotonic()-started04)/60:.1f} min",flush=True)
else:
    print("Direct download disabled. Set START_DIRECT04=True when ready.")

if START_EXPORTS04:
    try:ee.Initialize(project="musanga-511110")
    except Exception:
        ee.Authenticate()
        ee.Initialize(project="musanga-511110")
    tasks04=[]
    for site in EXPORT_SITES04:
        g=GRIDS04[site];year=int(g["uav_year"])
        for family in EXPORT_FAMILIES04:
            prefix=(f"{site}_S2_{year}_RAW_10m" if family=="RAW_ANNUAL"
                    else f"{site}_{family}_{year}_10m")
            # Unique staging folder per site/product; do not model from staging.
            folder=f"Musanga08_STAGE_{site}_{family}"
            task=ee.batch.Export.image.toDrive(
                image=create_feature_image04(site,family).unmask(-9999,sameFootprint=False),
                description=f"NB08_{prefix}",folder=folder,fileNamePrefix=prefix,
                region=ee_region04(site),crs=g["uav_crs"],
                crsTransform=g["target_transform"],
                maxPixels=1e10,fileFormat="GeoTIFF",
                formatOptions={"cloudOptimized":True,"noData":-9999})
            task.start()
            tasks04.append(dict(site=site,family=family,year=year,
                                task_id=task.id,folder=folder,prefix=prefix))
            print("Started",prefix,task.id)
    pd.DataFrame(tasks04).to_csv(VALIDATION/"04_new_export_task_manifest.csv",index=False)
else:
    print("DRY RUN — no new GEE tasks. Set START_EXPORTS04=True to create quarterly and embedding exports.")
    print("Annual RAW products already exist in",RAW_PROJECT04)


In [ ]:
# BLOCK 04B — Ploton et al. S2_AMAP tropical BRDF correction, ported from user's July 2026 PP GEE script
# Source: https://code.earthengine.google.com/bf3975028f60f6c5a699a44691f127e6
# This is NOT Roy c-factor: 9 AMAP coefficient sets, spatial orbit-based viewing
# geometry, pixel-wise astronomical solar geometry, HLS latitude-dependent reference sun.
# Harmonized S2_SR already applies baseline offsets: DO NOT subtract 1000 again.
# Run 00, 04, 04B. Extract all five sites with resumable existing-file verification.
# ================================================================
# EXECUTION SETTINGS — process all five sites, reuse existing files
# ================================================================
START_BRDF_DIRECT04B=True
BRDF_SITES04B=["MOS_1_2","Yoko","Yangambi","Baego","Luki"]
BRDF_FAMILIES04B=["S2_AMAP_QUARTERLY","S2_AMAP_T3Y"]  # 44 / 168 bands
# IMPORTANT: do not run any other large GEE downloads concurrently.
AMAP_THREADS04B=1  # sequential EE tile requests; minimizes 429 concurrency failures
AMAP_MAX_ATTEMPTS04B=4  # per quarterly checkpoint; exponential cooldown
AMAP_RETRY_BASE_SECONDS04B=45
# Fusion families are assembled later in Blocks 06-08, not downloaded.
# ================================================================
import ee,numpy as np,json,pandas as pd
# Initialize BEFORE constructing Earth Engine FeatureCollection objects.
try:
    ee.Initialize(project="musanga-511110")
except ee.EEException as exc:
    if "not initialized" in str(exc).lower() or "credentials" in str(exc).lower() or "authenticate" in str(exc).lower():
        ee.Authenticate()
        ee.Initialize(project="musanga-511110")
    else:
        raise
AMAP04B={
 "B2":(.0497,.0323,-.0021), "B3":(.0603,.0204,.0734),
 "B4":(.0258,-.0149,.1893), "B5":(.0856,.0173,.1397),
 "B6":(.2985,.1560,-.2528), "B7":(.3781,.1926,-.3409),
 "B8":(.3623,.1606,-.1811), "B11":(.2008,.0743,.1536),
 "B12":(.0901,.0334,.0957)}
AMAP_ORBITS04B="users/wiell/SepalResources/sentinel2TileOrbits"
ORBIT_LINES04B=ee.FeatureCollection(AMAP_ORBITS04B)
S2_BRDF04B=("S2_AMAP_QUARTERLY","S2_AMAP_T3Y")
FEATURES04["S2_AMAP_QUARTERLY"]=list(FEATURES04["S2_QUARTERLY"])
FEATURES04["S2_AMAP_T3Y"]=list(FEATURES04["S2_T3Y"])
FEATURES04["FUSION_AMAP"]=FEATURES04["RAW_ANNUAL"]+FEATURES04["S2_AMAP_QUARTERLY"]+FEATURES04["EMB_ANNUAL"]
FEATURES04["FUSION_AMAP_T3Y"]=FEATURES04["S2_AMAP_T3Y"]+FEATURES04["EMB_T3Y"]
def ex04B(expression,kwargs):
    return ee.Image().expression(expression,kwargs)
def rs04B(v):
    return ee.Image(v).multiply(np.pi/180)
def solar04B(date):
    # Literal port of PP getSunAngles(): astronomical solar azimuth and zenith
    lonlat=ee.Image.pixelLonLat()
    lat=rs04B(lonlat.select("latitude"))
    jdp=ee.Image(ee.Number(date.getFraction("year")).float())
    jdpr=jdp.multiply(2*np.pi)
    hour=ee.Number(date.getRelative("second","day")).divide(3600)
    mean=ee.Image(hour).add(lonlat.select("longitude").divide(15))
    diff=ex04B("(0.000075+0.001868*cos(j)-0.032077*sin(j)-0.014615*cos(2*j)-0.040849*sin(2*j))*12*60/PI",
               dict(j=jdpr,PI=np.pi))
    angle=rs04B(mean.add(diff.divide(60)).subtract(12).multiply(15))
    delta=ex04B("0.006918-0.399912*cos(j)+0.070257*sin(j)-0.006758*cos(2*j)+0.000907*sin(2*j)-0.002697*cos(3*j)+0.001480*sin(3*j)",dict(j=jdpr))
    zen=ex04B("acos(max(-1,min(1,sin(lat)*sin(d)+cos(lat)*cos(d)*cos(h))))",
               dict(lat=lat,d=delta,h=angle))
    sinaz=ex04B("cos(d)*sin(h)/sin(z)",dict(d=delta,h=angle,z=zen)).clamp(-1,1)
    cosaz=ex04B("(-cos(lat)*sin(d)+sin(lat)*cos(d)*cos(h))/sin(z)",
                dict(lat=lat,d=delta,h=angle,z=zen))
    az=sinaz.asin()
    az=az.where(cosaz.lte(0),ee.Image(np.pi).subtract(az))
    az=az.where(cosaz.gt(0).And(sinaz.lte(0)),ee.Image(2*np.pi).add(az))
    az=az.add(np.pi)
    az=az.where(az.gt(2*np.pi),az.subtract(2*np.pi))
    return az,zen
def target_sun04B(geometry):
    # HLS v1.2 degree-six latitude polynomial from PP's getSunZenOut.
    lat=ee.Number(geometry.centroid(30).coordinates().get(1))
    polynomial=ee.Number(31.0076).subtract(lat.multiply(.1272))
    for power,coef in [(2,.01187),(3,2.40e-5),(4,-9.48e-7),(5,-1.95e-9),(6,6.15e-11)]:
        polynomial=polynomial.add(lat.pow(power).multiply(coef))
    return ee.Image(polynomial.multiply(np.pi/180))
def orbit04B(image):
    fc=ORBIT_LINES04B.filter(ee.Filter.And(
        ee.Filter.eq("tileId",image.get("MGRS_TILE")),
        ee.Filter.eq("orbitNumber",image.get("SENSING_ORBIT_NUMBER"))))
    # The caller explicitly filters absent orbit records before mapping.
    rec=ee.Feature(fc.first())
    return ee.Geometry.LineString(ee.String(rec.get("orbit")).decodeJSON())
def view04B(image):
    # Literal geometric construction from PP getSentinel2ViewAngles().
    track=orbit04B(image)
    projected=track.transform("EPSG:3857",1)
    coords=projected.coordinates()
    p1=ee.List(coords.get(0));p2=ee.List(coords.get(ee.Number(coords.size()).subtract(1)))
    x1=ee.Number(p1.get(0));x2=ee.Number(p2.get(0))
    y1=ee.Number(p1.get(1));y2=ee.Number(p2.get(1))
    slope=y1.subtract(y2).divide(x1.subtract(x2))
    intercept=y1.subtract(slope.multiply(x1))
    dx=ee.Number(500).divide(ee.Number(1).add(ee.Number(1).divide(slope.pow(2))).sqrt())
    slope_perp=ee.Number(-1).divide(slope)
    def shifted(x):
        xx=x.add(dx)
        # Port of PP shiftLine: interceptPerp = slope*x + intercept - slopePerp*x
        yy=xx.multiply(slope_perp).add(slope.multiply(x).add(intercept).subtract(slope_perp.multiply(x)))
        return [xx,yy]
    shifted_track=ee.Geometry.LineString([shifted(x1),shifted(x2)],"EPSG:3857",False)
    left=ee.FeatureCollection([ee.Feature(shifted_track)])
    center=ee.FeatureCollection([ee.Feature(track)])
    dist_left=left.distance(150000,1)
    dist=center.distance(150000,1)
    signed=ee.Image(0).subtract(dist).where(dist.lt(dist_left).And(dist_left.gt(500)),dist)
    viewzen=signed.divide(786000).atan()
    viewaz=ee.Image(ee.Number(np.pi/2).subtract(slope_perp.atan()))
    return viewaz,viewzen
def kernels04B(s,v,phi):
    phase=ex04B("max(-1,min(1,cos(s)*cos(v)+sin(s)*sin(v)*cos(p)))",
                dict(s=s,v=v,p=phi))
    angle=phase.acos()
    kvol=ex04B("((PI/2-a)*c+sin(a))/(cos(s)+cos(v))-PI/4",
               dict(PI=np.pi,a=angle,c=phase,s=s,v=v))
    dist=ex04B("sqrt(max(0,pow(tan(s),2)+pow(tan(v),2)-2*tan(s)*tan(v)*cos(p)))",
               dict(s=s,v=v,p=phi))
    temp=ex04B("1/cos(s)+1/cos(v)",dict(s=s,v=v))
    cost=ex04B("2*sqrt(pow(d,2)+pow(tan(s)*tan(v)*sin(p),2))/t",
                dict(d=dist,s=s,v=v,p=phi,t=temp)).clamp(-1,1)
    t=cost.acos()
    overlap=ex04B("(1/PI)*(t-sin(t)*c)*den",dict(PI=np.pi,t=t,c=cost,den=temp))
    kgeo=ex04B("o-den+0.5*(1+phase)/(cos(s)*cos(v))",
               dict(o=overlap,den=temp,phase=phase,s=s,v=v))
    return kvol,kgeo
def corrected_scene04B(image):
    image=ee.Image(image)  # copyProperties returns generic Element in EE Python
    sunaz,sunzen=solar04B(ee.Date(image.get("system:time_start")))
    viewaz,viewzen=view04B(image)
    kvol,kgeo=kernels04B(sunzen,viewzen,sunaz.subtract(viewaz))
    refzen=target_sun04B(image.geometry())
    kvol0,kgeo0=kernels04B(refzen,ee.Image(0),ee.Image(0))
    corrected=[]
    for b in S2_BASE04:
        if b=="B8A":
            # AMAP table corrects nine bands, NOT B8A/redEdge4. Keep B8A
            # native; consequently NDRE has one uncorrected input.
            corrected.append(image.select(b))
            continue
        iso,geo,vol=AMAP04B[b]
        brdf=ee.Image(iso).add(kvol.multiply(vol)).add(kgeo.multiply(geo))
        brdf0=ee.Image(iso).add(kvol0.multiply(vol)).add(kgeo0.multiply(geo))
        corrected.append(image.select(b).multiply(brdf0.divide(brdf)).rename(b))
    return ee.Image(ee.Image.cat(corrected).copyProperties(image,["system:time_start"])
                    .set("AMAP_CORRECTED",1))
def amap_features04B(site,family,only_offset=None,only_quarter=None):
    year=int(GRIDS04[site]["uav_year"])
    geom=ee_region04(site)
    offsets=((0,) if family=="S2_AMAP_QUARTERLY" else (-1,0,1))
    if only_offset is not None:
        assert only_offset in offsets, (family,only_offset)
        offsets=(only_offset,)
    layers=[]
    for offset in offsets:
        yr=year+offset
        raw=(ee.ImageCollection(SRC04["s2"]).filterBounds(geom)
             .filterDate(f"{yr}-01-01",f"{yr+1}-01-01"))
        # Filter scenes without matching orbit geometries instead of using a
        # placeholder geometry that would silently create invalid corrections.
        def with_orbit(im):
            orbit_count=ORBIT_LINES04B.filter(ee.Filter.And(
                ee.Filter.eq("tileId",im.get("MGRS_TILE")),
                ee.Filter.eq("orbitNumber",im.get("SENSING_ORBIT_NUMBER")))).size()
            # Store numeric server-side count, not ee.Number.gt() as a property.
            # Earth Engine serializes boolean-like computed properties differently
            # from Python True, which can make eq(..., True) discard all scenes.
            return ee.Image(im).set("AMAP_ORBIT_MATCH_COUNT",orbit_count)
        indexed=raw.map(with_orbit)
        matched=indexed.filter(ee.Filter.gt("AMAP_ORBIT_MATCH_COUNT",0))
        corrected=matched.map(
            lambda im:corrected_scene04B(ee.Image(s2_mask04(im).copyProperties(im,im.propertyNames()))))
        for q in ((only_quarter,) if only_quarter is not None else range(1,5)):
            start=f"{yr}-{3*q-2:02d}-01"
            end=f"{yr+1}-01-01" if q==4 else f"{yr}-{3*q+1:02d}-01"
            part=corrected.filterDate(start,end)
            raw_n=int(raw.filterDate(start,end).size().getInfo())
            matched_n=int(matched.filterDate(start,end).size().getInfo())
            print(f"  AMAP orbit support {site} {yr} Q{q}: {matched_n}/{raw_n} scenes",flush=True)
            if matched_n==0:
                samples=indexed.filterDate(start,end).limit(4).aggregate_array("AMAP_ORBIT_MATCH_COUNT").getInfo()
                raise RuntimeError(f"{site} {yr} Q{q}: {matched_n}/{raw_n} orbit-matched scenes, "
                                   f"sample match counts={samples}; verify orbitNumber field type and image metadata.")
            if matched_n < raw_n:
                raise RuntimeError(f"{site} {yr} Q{q}: {raw_n-matched_n} S2 images lack orbit geometries; "
                                   "refusing to generate a biased partial-quarter AMAP composite.")
            med=part.median().multiply(.0001)
            if family=="S2_AMAP_T3Y":
                ix=ee.Image.cat([
                    med.normalizedDifference(["B8","B4"]).rename("NDVI"),
                    med.normalizedDifference(["B8","B11"]).rename("NDMI"),
                    med.normalizedDifference(["B8","B12"]).rename("NBR"),
                    med.normalizedDifference(["B8A","B5"]).rename("NDRE")])
                layers.append(ee.Image.cat([med,ix]).rename([
                    f"{YEAR_OFFSET_TAG04[offset]}_Q{q}_{b}" for b in S2_BASE04+S2_INDICES04]))
            else:
                layers.append(med.rename([f"Q{q}_{b}" for b in S2_BASE04]))
                layers.append(part.select("B2").count().unmask(0).toFloat().rename(f"Q{q}_N_CLEAR"))
    return ee.Image.cat(layers).toFloat().clip(geom)
create_feature_image04_AMAP_original=create_feature_image04
def create_feature_image04(site,family):
    if family in S2_BRDF04B:return amap_features04B(site,family)
    return create_feature_image04_AMAP_original(site,family)
BRDF_CONTRACT04B=dict(method="Ploton_S2_AMAP_July2026_script",
    source="https://code.earthengine.google.com/bf3975028f60f6c5a699a44691f127e6",
    orbit_lines=AMAP_ORBITS04B,coefficients=AMAP04B,
    solar="pixelwise PP astronomical",view="spatial tile-orbit track PP method",
    reference_sun="HLS latitude polynomial",baseline="S2_SR_HARMONIZED already harmonized",
    bands_unadjusted=["B8A"],composite="correct individual scenes before quarterly median")
(VALIDATION/"04B_amap_contract.json").write_text(json.dumps(BRDF_CONTRACT04B,indent=2))
display(pd.DataFrame([dict(family=f,bands=len(FEATURES04[f])) for f in
                     ("S2_AMAP_QUARTERLY","S2_AMAP_T3Y","FUSION_AMAP","FUSION_AMAP_T3Y")]))

# ── Resumable 14-band quarterly GeoTIFFs → 56-band annual → 168-band final ──
def download_amap_t3y_chunked04B(site):
    import os,shutil,time,random,logging
    import rasterio,numpy as np,geemap
    from pathlib import Path
    from affine import Affine
    from rasterio.windows import Window
    g=GRIDS04[site]; year=int(g["uav_year"])
    family="S2_AMAP_T3Y"; folder=EXTRA_PROJECT04/family
    folder.mkdir(parents=True,exist_ok=True)
    final=folder/f"{site}_{family}_{year}_10m.tif"
    expected=Affine(*g["target_transform"])
    checkpoint_dir=folder/"_checkpoints"/site
    checkpoint_dir.mkdir(parents=True,exist_ok=True)
    def verify(path,bands,names=None):
        with rasterio.open(path) as ds:
            assert ds.count==bands,(path,"bands",ds.count,bands)
            assert ds.width==g["width"] and ds.height==g["height"],(path,"grid")
            assert str(ds.crs)==g["uav_crs"],(path,"crs")
            assert ds.transform.almost_equals(expected,precision=8),(path,"transform")
            assert ds.nodata==-9999,(path,"nodata",ds.nodata)
            assert ds.dtypes[0]=="float32",(path,"dtype")
            if names is not None:
                assert list(ds.descriptions)==list(names),(path,"band order/labels")
            for k in range(1,bands+1):
                for _,win in ds.block_windows(k):
                    arr=ds.read(k,window=win)
                    assert np.isfinite(arr).all(),(path,k,"nonfinite")
        return True
    if final.exists():
        verify(final,168,FEATURES04[family])
        print("VERIFIED existing 168-band AMAP:",final,flush=True)
        return
    def publish(target,write_fn,bands,names):
        # Do not touch existing completed checkpoints; publish atomically
        incoming=target.with_suffix(".incoming")
        if incoming.exists():
            print("Removing incomplete staging file:",incoming,flush=True)
            incoming.unlink()
        try:
            write_fn(incoming)
            verify(incoming,bands,names)
            os.replace(incoming,target)
        except BaseException:
            if incoming.exists():incoming.unlink()
            raise
    def write_stack(srcs,dst,names,tags):
        with rasterio.open(srcs[0]) as ds:
            profile=ds.profile.copy()
        profile.update(driver="GTiff",count=len(names),dtype="float32",
                       nodata=-9999.0,tiled=True,compress="deflate",
                       predictor=3,bigtiff="IF_SAFER")
        with rasterio.open(dst,"w",**profile) as out:
            j=1
            for src in srcs:
                with rasterio.open(src) as ds:
                    for band in range(1,ds.count+1):
                        for _,win in ds.block_windows(band):
                            out.write(ds.read(band,window=win),j,window=win)
                        out.set_band_description(j,names[j-1])
                        j+=1
            out.update_tags(**tags)
    annual_paths=[]
    for off in (-1,0,1):
        target_year=year+off
        year_tag=YEAR_OFFSET_TAG04[off]
        annual_names=[f"{year_tag}_Q{q}_{b}" for q in range(1,5)
                      for b in S2_BASE04+S2_INDICES04]
        annual_path=checkpoint_dir/f"{site}_AMAP_{target_year}_56bands.tif"
        if annual_path.exists():
            verify(annual_path,56,annual_names)
            print(f"  VERIFIED annual checkpoint {target_year}: {annual_path.name}",flush=True)
            annual_paths.append(annual_path)
            continue
        qpaths=[]
        for q in range(1,5):
            qnames=[f"{year_tag}_Q{q}_{b}" for b in S2_BASE04+S2_INDICES04]
            qpath=checkpoint_dir/f"{site}_AMAP_{target_year}_Q{q}_14bands.tif"
            if qpath.exists():
                verify(qpath,14,qnames)
                print(f"  VERIFIED quarter {target_year} Q{q}, skipped",flush=True)
                qpaths.append(qpath)
                continue
            scratch=Path("/content")/f"amap04B_{site}_{target_year}_Q{q}.tif"
            # An interrupted scratch is not a verified checkpoint.
            if scratch.exists():scratch.unlink()
            print(f"  START {site} {target_year} Q{q}: 14 bands, 1 worker",flush=True)
            # Build only 14 bands of the quarter; do not evaluate other quarters.
            image=amap_features04B(site,family,only_offset=off,only_quarter=q)
            image=image.unmask(-9999,sameFootprint=False).toFloat()
            for attempt in range(1,AMAP_MAX_ATTEMPTS04B+1):
                try:
                    if scratch.exists():scratch.unlink()
                    geemap.download_ee_image(
                        image,filename=str(scratch),crs=g["uav_crs"],
                        crs_transform=g["target_transform"],
                        shape=(g["height"],g["width"]),dtype="float32",
                        resampling="near",max_tile_dim=256,
                        num_threads=AMAP_THREADS04B,overwrite=True)
                    with rasterio.open(scratch,"r+") as ds:
                        # geedim may advertise -inf as NoData while pixels
                        # contain the requested finite -9999 sentinel.
                        assert ds.count==14 and ds.dtypes[0]=="float32"
                        for k in range(1,15):
                            for _,win in ds.block_windows(k):
                                assert np.isfinite(ds.read(k,window=win)).all()
                        if ds.nodata is None or (np.isinf(ds.nodata) and ds.nodata<0):
                            ds.nodata=-9999.0
                        assert ds.nodata==-9999.0
                        for k,name in enumerate(qnames,1):
                            ds.set_band_description(k,name)
                        ds.update_tags(FAMILY=family,BRDF_METHOD="Ploton_S2_AMAP_July2026",
                                       REFERENCE_YEAR=str(year),S2_YEAR=str(target_year),QUARTER=str(q))
                    verify(scratch,14,qnames)
                    publish(qpath,lambda dst:shutil.copyfile(scratch,dst),14,qnames)
                    print(f"  SAVED quarter {target_year} Q{q}: {qpath.name}",flush=True)
                    scratch.unlink()
                    qpaths.append(qpath)
                    break
                except Exception as e:
                    if scratch.exists():scratch.unlink()
                    if attempt==AMAP_MAX_ATTEMPTS04B:
                        raise RuntimeError(
                            f"{site} {target_year} Q{q}: failed after {attempt} attempts; "
                            "previous Drive checkpoints remain safe") from e
                    pause=AMAP_RETRY_BASE_SECONDS04B*(2**(attempt-1))
                    print(f"  RETRY {site} {target_year} Q{q} attempt {attempt}/{AMAP_MAX_ATTEMPTS04B}: "
                          f"{type(e).__name__}: {str(e)[:220]}; cooling down {pause}s",flush=True)
                    time.sleep(pause)
        print(f"  ASSEMBLING annual {target_year} from 4 validated quarterly files...",flush=True)
        publish(annual_path,lambda dst:write_stack(
            qpaths,dst,annual_names,
            dict(FAMILY=family,BRDF_METHOD="Ploton_S2_AMAP_July2026",
                 REFERENCE_YEAR=str(year),S2_YEAR=str(target_year))),56,annual_names)
        print("  SAVED annual checkpoint:",annual_path,flush=True)
        annual_paths.append(annual_path)
    print(f"  ASSEMBLING final 168-band {site} from 3 annual files...",flush=True)
    publish(final,lambda dst:write_stack(
        annual_paths,dst,FEATURES04[family],
        dict(FAMILY=family,BRDF_METHOD="Ploton_S2_AMAP_July2026",
             ACQUISITION_YEAR=str(year),TEMPORAL="year-1/year/year+1",
             RAW_S2_NO_BRDF="NO")),168,FEATURES04[family])
    print("SAVED COMPLETE final AMAP T3Y:",final,flush=True)

if START_BRDF_DIRECT04B:
    try:ee.Initialize(project="musanga-511110")
    except Exception:
        ee.Authenticate()
        ee.Initialize(project="musanga-511110")
    print("Checking access to PP's Sentinel-2 tile/orbit asset:",AMAP_ORBITS04B)
    print("Orbit records available:",ORBIT_LINES04B.limit(1).size().getInfo())
    for site in BRDF_SITES04B:
        raw=(ee.ImageCollection(SRC04["s2"]).filterBounds(ee_region04(site))
             .filterDate(f"{int(GRIDS04[site]['uav_year'])}-01-01",
                         f"{int(GRIDS04[site]['uav_year'])+1}-01-01"))
        # Require actual orbit coverage for the acquisition-year scenes.
        tile_orbit_keys=raw.aggregate_array("MGRS_TILE").getInfo()
        orbit_numbers=raw.aggregate_array("SENSING_ORBIT_NUMBER").getInfo()
        distinct_pairs=sorted({(t,int(o)) for t,o in zip(tile_orbit_keys,orbit_numbers) if t is not None and o is not None})
        print("Unique tile/orbit pairs:",len(distinct_pairs))
        for tile,orbit in distinct_pairs:
            hits=ORBIT_LINES04B.filter(ee.Filter.And(ee.Filter.eq("tileId",tile),ee.Filter.eq("orbitNumber",orbit))).size().getInfo()
            if hits==0:raise RuntimeError(f"AMAP orbit geometry missing for {site}: {tile} orbit {orbit}; refusing partial correction")
        first=ee.Image(raw.first())
        key=first.getInfo().get("properties",{})
        print("Smoke-test site:",site,"MGRS tile",key.get("MGRS_TILE"),
              "orbit",key.get("SENSING_ORBIT_NUMBER"))
        for family in BRDF_FAMILIES04B:
            print("AMAP corrected extraction:",site,family,flush=True)
            if family=="S2_AMAP_T3Y":
                download_amap_t3y_chunked04B(site)
            else:
                download_direct04(site,family)

else:
    print("Ploton S2_AMAP implementation registered. No BRDF extraction yet.")
    print("For first run set START_BRDF_DIRECT04B=True, keep BRDF_SITES04B=['Yoko'].")


In [ ]:
# BLOCK 04C — STATIC QA: one raw Sentinel-2 scene versus Ploton S2_AMAP
# Run after 00, 04, 04B. Does NOT launch new quarterly/three-year exports.
# Uses identical scene correction function to the production pipeline.
# ================================================================
# SETTINGS — change only these for another site/date/band
# ================================================================
QA_SITE04C="Yoko"
QA_YEAR04C=2020
QA_QUARTER04C=3
QA_BAND04C="B8"                  # AMAP-corrected: B2,B3,B4,B5,B6,B7,B8,B11,B12
QA_SCENE_INDEX04C=0             # rank by CLOUDY_PIXEL_PERCENTAGE
QA_MAX_SCENES04C=12
QA_RATIO_RANGE04C=(0.85,1.15)
QA_SAVE_PNG04C=True
# ================================================================
import ee, geemap, rasterio, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
assert QA_BAND04C in AMAP04B, "Only the 9 AMAP-corrected bands are valid for this diagnostic."
assert callable(corrected_scene04B) and callable(s2_mask04)
g04c=GRIDS04[QA_SITE04C]
geom04c=ee_region04(QA_SITE04C)
q04c=QA_QUARTER04C
start04c=f"{QA_YEAR04C}-{3*q04c-2:02d}-01"
end04c=(f"{QA_YEAR04C+1}-01-01" if q04c==4
         else f"{QA_YEAR04C}-{3*q04c+1:02d}-01")
coll04c=(ee.ImageCollection(SRC04["s2"])
         .filterBounds(geom04c).filterDate(start04c,end04c)
         .sort("CLOUDY_PIXEL_PERCENTAGE"))
count04c=int(coll04c.size().getInfo())
assert count04c>QA_SCENE_INDEX04C,(count04c,QA_SCENE_INDEX04C)
meta04c=coll04c.limit(min(QA_MAX_SCENES04C,count04c))
print("Scene candidates (cloud % refers to entire tile, not necessarily Yoko):")
display(__import__("pandas").DataFrame({
    "scene_id":meta04c.aggregate_array("system:index").getInfo(),
    "tile":meta04c.aggregate_array("MGRS_TILE").getInfo(),
    "orbit":meta04c.aggregate_array("SENSING_ORBIT_NUMBER").getInfo(),
    "tile_cloud_percent":meta04c.aggregate_array("CLOUDY_PIXEL_PERCENTAGE").getInfo()
}))
scene04c=ee.Image(coll04c.toList(QA_SCENE_INDEX04C+1).get(QA_SCENE_INDEX04C))
props04c=scene04c.toDictionary(["system:index","MGRS_TILE","SENSING_ORBIT_NUMBER","CLOUDY_PIXEL_PERCENTAGE"]).getInfo()
hits04c=int(ORBIT_LINES04B.filter(ee.Filter.And(
    ee.Filter.eq("tileId",props04c["MGRS_TILE"]),
    ee.Filter.eq("orbitNumber",props04c["SENSING_ORBIT_NUMBER"]))).size().getInfo())
assert hits04c>0,("No AMAP orbit geometry for this scene",props04c)
print("Chosen scene:",props04c)
# Exactly the same cloud screening and image correction as production Block 04B.
clean04c=ee.Image(s2_mask04(scene04c).copyProperties(scene04c,scene04c.propertyNames()))
raw04c=clean04c.select(QA_BAND04C).multiply(.0001).rename("RAW")
corrected04c=corrected_scene04B(clean04c).select(QA_BAND04C).multiply(.0001).rename("AMAP")
# c-factor is independent of reflectance (except division at raw=0).
good04c=raw04c.mask().And(corrected04c.mask()).And(raw04c.abs().gt(1e-6))
raw04c=raw04c.updateMask(good04c)
corrected04c=corrected04c.updateMask(good04c)
factor04c=corrected04c.divide(raw04c).rename("FACTOR")
delta04c=corrected04c.subtract(raw04c).rename("DELTA")
image04c=ee.Image.cat([raw04c,corrected04c,factor04c,delta04c]).clip(geom04c)
scratch04c=Path("/content")/f"musanga_04C_{QA_SITE04C}_{QA_YEAR04C}_Q{q04c}_{QA_BAND04C}_scene{QA_SCENE_INDEX04C}.tif"
if scratch04c.exists(): scratch04c.unlink() # small temporary QA only; avoid stale scene selection
geemap.download_ee_image(
    image04c.unmask(-9999,sameFootprint=False).toFloat(),
    filename=str(scratch04c),
    crs=g04c["uav_crs"],crs_transform=g04c["target_transform"],
    shape=(g04c["height"],g04c["width"]),
    dtype="float32",resampling="near",max_tile_dim=256,num_threads=2,
    overwrite=True)
with rasterio.open(scratch04c) as ds:
    stack04c=ds.read().astype(float)
stack04c[stack04c==-9999]=np.nan
raw_np04c,cor_np04c,ratio_np04c,diff_np04c=stack04c
v04c=np.isfinite(raw_np04c)&np.isfinite(cor_np04c)&np.isfinite(ratio_np04c)
assert v04c.any(),"No valid pixels for this selected scene; try QA_SCENE_INDEX04C=1 or another quarter."
vmin04c,vmax04c=np.nanpercentile(np.concatenate((raw_np04c[v04c],cor_np04c[v04c])),[2,98])
dlim04c=float(np.nanpercentile(np.abs(diff_np04c[v04c]),98))
fig04c,axs04c=plt.subplots(1,4,figsize=(19,5),constrained_layout=True)
spec04c=[
 (raw_np04c,f"RAW {QA_BAND04C}","viridis",vmin04c,vmax04c),
 (cor_np04c,f"AMAP {QA_BAND04C}","viridis",vmin04c,vmax04c),
 (ratio_np04c,"Correction factor (AMAP / RAW)","RdBu_r",*QA_RATIO_RANGE04C),
 (diff_np04c,"Corrected - RAW","RdBu_r",-dlim04c,dlim04c)]
for ax,(a,title,cmap,lo,hi) in zip(axs04c,spec04c):
    im=ax.imshow(a,cmap=cmap,vmin=lo,vmax=hi,interpolation="nearest")
    ax.set_title(title,fontsize=11);ax.axis("off")
    fig04c.colorbar(im,ax=ax,fraction=.045,pad=.025)
fig04c.suptitle(f"{QA_SITE04C} | single S2 scene | {QA_YEAR04C} Q{q04c} | {props04c['system:index']}",fontsize=12)
if QA_SAVE_PNG04C:
    png04c=OUT/"validation"/f"04C_{QA_SITE04C}_{QA_YEAR04C}_Q{q04c}_{QA_BAND04C}_scene{QA_SCENE_INDEX04C}.png"
    png04c.parent.mkdir(parents=True,exist_ok=True)
    fig04c.savefig(png04c,dpi=160,bbox_inches="tight")
    print("Saved figure:",png04c)
plt.show()
print("Scene QA:",dict(n_valid=int(v04c.sum()),
    ratio_median=float(np.nanmedian(ratio_np04c[v04c])),
    ratio_p05=float(np.nanpercentile(ratio_np04c[v04c],5)),
    ratio_p95=float(np.nanpercentile(ratio_np04c[v04c],95)),
    delta_median=float(np.nanmedian(diff_np04c[v04c]))))


## 05 — Import and align satellite features in the **Musanga project**

Reads previously consolidated RAW rasters from `satellite_raw_s2_10m/`; imports completed **S2_QUARTERLY** and **EMB_ANNUAL** exports from *site+family-specific* Earth Engine staging folders into `satellite_features/<family>/`. Every raster is checked for band count, CRS, 10 m affine transform, NoData and pixel origin. When Earth Engine adds padded rows or columns, crop/pad by **integer pixel shifts only**, never interpolate reflectance or embeddings. Validates SHA256 on exact copies; preserves source staging files. Writes `validation/05_canonical_feature_manifest.csv`.

Block 05 can run as soon as RAW files exist; additional features appear automatically after exports finish.

In [ ]:
# BLOCK 05 — Canonical feature import (RAW already done; new features staged by EE)
# Run 00 + 04. No Earth Engine calls. Never delete source files.
import os,json,shutil,hashlib
from pathlib import Path
import pandas as pd,numpy as np,rasterio
from affine import Affine
from rasterio.windows import Window
from IPython.display import display
assert "FEATURES04" in globals()
CANON05={"RAW_ANNUAL":RAW_PROJECT04,
         "S2_QUARTERLY":EXTRA_PROJECT04/"S2_QUARTERLY",
         "EMB_ANNUAL":EXTRA_PROJECT04/"EMB_ANNUAL",
         "S2_T3Y":EXTRA_PROJECT04/"S2_T3Y",
         "EMB_T3Y":EXTRA_PROJECT04/"EMB_T3Y",
         "S2_BRDF_QUARTERLY":EXTRA_PROJECT04/"S2_BRDF_QUARTERLY",
         "S2_BRDF_T3Y":EXTRA_PROJECT04/"S2_BRDF_T3Y",
         "S2_AMAP_QUARTERLY":EXTRA_PROJECT04/"S2_AMAP_QUARTERLY",
         "S2_AMAP_T3Y":EXTRA_PROJECT04/"S2_AMAP_T3Y"}
for folder in CANON05.values():folder.mkdir(parents=True,exist_ok=True)
def sha05(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b""):h.update(chunk)
    return h.hexdigest()
def inspect05(p,site,family,require_exact=False):
    g=GRIDS04[site];target=Affine(*g["target_transform"])
    with rasterio.open(p) as im:
        if im.count!=len(FEATURES04[family]):raise RuntimeError(f"{p}: expected {len(FEATURES04[family])} bands; got {im.count}")
        if str(im.crs)!=g["uav_crs"]:raise RuntimeError(f"{p}: CRS {im.crs} != {g['uav_crs']}")
        if im.nodata!=-9999:raise RuntimeError(f"{p}: NoData {im.nodata} != -9999")
        a=im.transform
        if not(np.isclose(a.a,10) and np.isclose(a.e,-10) and abs(a.b)<1e-8 and abs(a.d)<1e-8):
            raise RuntimeError(f"{p}: not a nonrotated 10m grid: {a}")
        col=(target.c-a.c)/a.a
        row=(target.f-a.f)/a.e
        if max(abs(col-round(col)),abs(row-round(row)))>1e-6:
            raise RuntimeError(f"{p}: pixel grids NOT integer aligned: {a} vs {target}")
        exact=(im.width==int(g["width"]) and im.height==int(g["height"]) and a.almost_equals(target))
        if require_exact and not exact:raise RuntimeError(f"{p}: canonical grid mismatch")
        return dict(exact=exact,row=int(round(row)),col=int(round(col)),
                    source_shape=(im.height,im.width))
def stage_folders05(site,family):
    key=f"Musanga08_STAGE_{site}_{family}"
    return [p for p in MY.iterdir() if p.is_dir() and
            (p.name==key or p.name.startswith(key+" ("))]
def write_aligned05(srcpath,tmp,site,family,info):
    g=GRIDS04[site]
    with rasterio.open(srcpath) as source:
        profile=source.profile.copy()
        profile.update(driver="GTiff",count=len(FEATURES04[family]),
                       width=int(g["width"]),height=int(g["height"]),
                       transform=Affine(*g["target_transform"]),nodata=-9999,
                       tiled=True,blockxsize=256,blockysize=256,
                       compress="deflate")
        profile.pop("predictor",None)
        with rasterio.open(tmp,"w",**profile) as dest:
            for band in range(1,source.count+1):
                dest.set_band_description(band,FEATURES04[family][band-1])
            for _,w in dest.block_windows(1):
                sw=Window(w.col_off+info["col"],w.row_off+info["row"],
                          w.width,w.height)
                dest.write(source.read(window=sw,boundless=True,fill_value=-9999),
                           window=w)
            dest.update_tags(SOURCE_EXPORT=str(srcpath),ALIGNMENT="INTEGER_PIXEL_CROP_PAD_NO_RESAMPLING")
manifest05=[]
for site in SITES:
    y=int(GRIDS04[site]["uav_year"])
    for family in ["RAW_ANNUAL","S2_QUARTERLY","EMB_ANNUAL","S2_T3Y","EMB_T3Y"]:
        prefix=(f"{site}_S2_{y}_RAW_10m" if family=="RAW_ANNUAL"
                else f"{site}_{family}_{y}_10m")
        dest=CANON05[family]/(prefix+".tif")
        if dest.is_file():
            info=inspect05(dest,site,family,require_exact=True)
            manifest05.append(dict(site=site,family=family,year=y,
                status="VERIFIED",file=str(dest),sha256=sha05(dest),
                alignment="EXACT_CANONICAL"))
            print(site,family,"VERIFIED",dest.name)
            continue
        if family=="RAW_ANNUAL":
            print(site,"RAW_ANNUAL missing canonical file; use the previously validated RAW import or restore from Git history.")
            continue
        candidates=[]
        for folder in stage_folders05(site,family):
            candidates += [v for v in folder.glob(prefix+"*.tif") if v.is_file()]
            candidates += [v for v in folder.glob(prefix+"*.tiff") if v.is_file()]
        if not candidates:
            print(site,family,"WAITING: export not yet present (or not submitted)")
            continue
        if len(candidates)>1:
            hashes={str(p):sha05(p) for p in candidates}
            if len(set(hashes.values()))!=1:
                raise RuntimeError(f"{site} {family}: conflicting staging exports: {hashes}")
        src=candidates[0]
        info=inspect05(src,site,family)
        tmp=dest.with_suffix(".incoming")
        if tmp.exists():
            raise RuntimeError(f"Staging file exists; inspect before rerun: {tmp}")
        try:
            if info["exact"]:
                shutil.copyfile(src,tmp)
                mode="BYTE_EXACT_COPY"
                if sha05(src)!=sha05(tmp):raise RuntimeError(f"{site}: checksum mismatch")
            else:
                write_aligned05(src,tmp,site,family,info)
                mode="PIXEL_ALIGNED_CROP_PAD"
            inspect05(tmp,site,family,require_exact=True)
            os.replace(tmp,dest)
        except Exception:
            tmp.unlink(missing_ok=True)
            raise
        manifest05.append(dict(site=site,family=family,year=y,
              status="IMPORTED",file=str(dest),sha256=sha05(dest),
              source_file=str(src),source_sha256=sha05(src),alignment=mode))
        print(site,family,mode)
MANIFEST05=pd.DataFrame(manifest05)
MANIFEST05.to_csv(VALIDATION/"05_canonical_feature_manifest.csv",index=False)
display(MANIFEST05)
print("Imported/verified",len(MANIFEST05),"physical feature products out of 15.")
print("No source staging files were deleted. Fusion is generated virtually in Stage 07.")


## 06 — Aligned predictors, target quality and **common support**

A feature family enters the benchmark **only when available at all five sites**. Builds one strict common-support mask per site across every available physical family, preventing models from being compared on different target pixels. Reference cover threshold and 90% UAV-valid rule remain unchanged; no train/test leakage.

In [ ]:
# BLOCK 06 — Join canonical predictor files and frozen UAV target grids; QA
# Run 00,04,05. Does NOT fit models. Optional families are checked when complete.
import numpy as np,pandas as pd,rasterio
from pathlib import Path
from affine import Affine
from IPython.display import display
assert "CANON05" in globals() and "FEATURES04" in globals()
TARGET06={s:OUT/"uav_targets_10m"/f"{s}_musanga_uav_target_S2grid_10m.tif" for s in SITES}
FEATURE_FILES06={}
ACTIVE06=[]
for family in ["RAW_ANNUAL","S2_QUARTERLY","EMB_ANNUAL","S2_T3Y","EMB_T3Y","S2_AMAP_QUARTERLY","S2_AMAP_T3Y"]:
    mapping={}
    for site in SITES:
        y=int(GRIDS04[site]["uav_year"])
        stem=(f"{site}_S2_{y}_RAW_10m" if family=="RAW_ANNUAL"
              else f"{site}_{family}_{y}_10m")
        fp=CANON05[family]/(stem+".tif")
        if fp.exists():
            inspect05(fp,site,family,require_exact=True)
            mapping[site]=fp
    if len(mapping)==len(SITES):
        ACTIVE06.append(family)
        FEATURE_FILES06[family]=mapping
    else:
        print(f"{family}: {len(mapping)}/5 sites available; excluded from comparative benchmark.",
              "missing:",[s for s in SITES if s not in mapping])
assert "RAW_ANNUAL" in ACTIVE06,"RAW baseline must be available at all 5 sites"
if "S2_QUARTERLY" in ACTIVE06 and "EMB_ANNUAL" in ACTIVE06:
    ACTIVE06.append("FUSION")
if "S2_T3Y" in ACTIVE06 and "EMB_T3Y" in ACTIVE06:
    ACTIVE06.append("FUSION_T3Y")
if "S2_AMAP_QUARTERLY" in ACTIVE06 and "EMB_ANNUAL" in ACTIVE06:
    ACTIVE06.append("FUSION_AMAP")
if "S2_AMAP_T3Y" in ACTIVE06 and "EMB_T3Y" in ACTIVE06:
    ACTIVE06.append("FUSION_AMAP_T3Y")
qa06=[]
band_qa06=[]
family_qa06=[]
for site in SITES:
    tgt=TARGET06[site]
    if not tgt.exists():raise FileNotFoundError(tgt)
    with rasterio.open(tgt) as im:
        assert im.count==4 and im.nodata==-9999 and im.tags().get("COMPLETE")=="YES"
        assert str(im.crs)==GRIDS04[site]["uav_crs"]
        assert im.transform.almost_equals(Affine(*GRIDS04[site]["target_transform"]))
        valid=(im.read(3)>=.90-1e-5)&(im.read(2)!=-9999)
    satellite_valid={}
    for f in ACTIVE06:
        # Fusions are virtual concatenations, not physical GeoTIFFs.
        if f in ("FUSION","FUSION_T3Y","FUSION_BRDF","FUSION_BRDF_T3Y","FUSION_AMAP","FUSION_AMAP_T3Y"):continue
        with rasterio.open(FEATURE_FILES06[f][site]) as im:
            arr=im.read()
            satellite_valid[f]=(np.isfinite(arr)&(arr!=-9999)).all(axis=0)
            if f in ("RAW_ANNUAL","S2_QUARTERLY","S2_BRDF_QUARTERLY","S2_AMAP_QUARTERLY"):
                counts=arr[14] if f=="RAW_ANNUAL" else arr[[10,21,32,43]].min(axis=0)
                satellite_valid[f] &= (counts>0)
            # Diagnose how much valid UAV reference survives EACH physical
            # feature family, before intersection across all datasets.
            family_qa06.append(dict(site=site,family=f,uav_reference=int(valid.sum()),
                joint_uav_feature=int((valid & satellite_valid[f]).sum()),
                fraction_uav_retained=float((valid & satellite_valid[f]).sum()/max(valid.sum(),1))))
            bandvalid=np.isfinite(arr)&(arr!=-9999)
            n_reference=max(int(valid.sum()),1)
            for j,name in enumerate(FEATURES04[f]):
                n_band=int((valid & bandvalid[j]).sum())
                band_qa06.append(dict(site=site,family=f,band_index=j+1,band=name,
                    valid_at_uav=n_band,uav_reference=int(valid.sum()),
                    missing_fraction_at_uav=1-n_band/n_reference))
    common=valid.copy()
    for mask in satellite_valid.values():common &= mask
    qa06.append(dict(site=site,uav_reference=int(valid.sum()),
        common_all_available_feature_families=int(common.sum()),
        raw_valid=int(satellite_valid["RAW_ANNUAL"].sum()),
        missing_common=int((valid&~common).sum()),
        families="+".join(ACTIVE06)))
QA06=pd.DataFrame(qa06)
QA06.to_csv(VALIDATION/"06_feature_common_support_QA.csv",index=False)
display(QA06)
print("Available benchmark families:",ACTIVE06)
print("IMPORTANT: Blocks 07–08 use FAMILY-SPECIFIC pixels, not the all-family common mask.")
print("For RAW-vs-AMAP paired comparisons, match pixels explicitly within each scale/site.")

FAMILY_COVERAGE06=pd.DataFrame(family_qa06)
BAND_COVERAGE06=pd.DataFrame(band_qa06)
FAMILY_COVERAGE06.to_csv(VALIDATION/"06_family_specific_uav_support.csv",index=False)
BAND_COVERAGE06.to_csv(VALIDATION/"06_band_specific_uav_support.csv",index=False)
print("\nIndividual feature-family support at UAV reference pixels:")
display(FAMILY_COVERAGE06.pivot(index="site",columns="family",values="joint_uav_feature"))
print("\nWorst 15 individual bands for Luki (at existing valid UAV pixels):")
display(BAND_COVERAGE06[BAND_COVERAGE06.site=="Luki"].sort_values("missing_fraction_at_uav",ascending=False).head(15))
if (QA06["common_all_available_feature_families"]<1000).any():
    print("QA NOTE: all-family intersection contains <1000 pixels at some sites, notably Luki.")
    print("That common mask is DIAGNOSTIC ONLY and is NOT used in Blocks 07–08.")
    print("Luki three-year S2 (raw and AMAP) still has too little support for a representative LOSO estimate.")
    print("Inspect 06_family_specific_uav_support.csv before interpreting those families.")


## 07 — **Single matched-sample, five-site LOSO experiment**

Uses a fixed HistGradientBoosting regression configuration, site-balanced training (up to 6,000 cells/site) and 20,000 held-out cells/site. Each available feature family is evaluated on **identical reference samples for that site**, including the four-site training pool and test pixels. Benchmarks both mean U-Net probability and threshold-derived canopy fraction against the training-mean baseline. Saves per-site metrics and Luki prediction vectors to `feature_family_benchmark/`.

**Interpretation:** This is a preliminary fixed-model test, not a nested or model-tuned evaluation. Spatially correlated cells do not constitute independent replicates. Luki overprediction in prior RAW baseline is an important transfer diagnostic. Running now (with only RAW available) reproduces the 15-band baseline; rerun when seasonal S2 and embeddings have been imported to compare all four feature families.

In [ ]:
# BLOCK 07 — ONE matched-sample 5-site LOSO benchmark for all feature families
# Run 00,04,05,06. CPU, no GEE, no raster modifications.
import numpy as np,pandas as pd,rasterio
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
from IPython.display import display
MAX_TRAIN07=6000
MAX_TEST07=20000
SEED07=20261011
OUTPUT07=OUT/"feature_family_benchmark"
OUTPUT07.mkdir(parents=True,exist_ok=True)
def systematic_locations07(rows,cols,limit,seed):
    # Deterministic spatially spread 10m cells; does not imply independent samples.
    order=np.lexsort((cols,rows))
    if len(order)<=limit:return rows[order],cols[order]
    rng=np.random.default_rng(seed)
    groups=np.array_split(order,limit)
    chosen=np.array([g[rng.integers(len(g))] for g in groups],dtype=int)
    return rows[chosen],cols[chosen]
def load_site07(site,family,limit,seed):
    with rasterio.open(TARGET06[site]) as ds:
        target=ds.read([1,2,3])
    mask=(np.isfinite(target).all(axis=0)&(target[0]!=-9999)&
          (target[1]!=-9999)&(target[2]>=.90-1e-5))
    arrays={}
    required={"FUSION":["RAW_ANNUAL","S2_QUARTERLY","EMB_ANNUAL"],"FUSION_T3Y":["S2_T3Y","EMB_T3Y"],"FUSION_BRDF":["RAW_ANNUAL","S2_BRDF_QUARTERLY","EMB_ANNUAL"],"FUSION_BRDF_T3Y":["S2_BRDF_T3Y","EMB_T3Y"],"FUSION_AMAP":["RAW_ANNUAL","S2_AMAP_QUARTERLY","EMB_ANNUAL"],"FUSION_AMAP_T3Y":["S2_AMAP_T3Y","EMB_T3Y"]}.get(family,[family])
    for family_part in required:
        with rasterio.open(FEATURE_FILES06[family_part][site]) as ds:
            a=ds.read().astype(np.float32)
        good=(np.isfinite(a)&(a!=-9999)).all(axis=0)
        if family_part=="RAW_ANNUAL":good &= (a[14]>0)
        if family_part in ("S2_QUARTERLY","S2_BRDF_QUARTERLY","S2_AMAP_QUARTERLY"):good &= (a[[10,21,32,43]]>0).all(axis=0)
        mask &= good
        arrays[family_part]=a
    rr,cc=np.where(mask)
    if not len(rr):raise RuntimeError(f"{site}: no common valid cells for {ACTIVE06}")
    rr,cc=systematic_locations07(rr,cc,limit,seed)
    X={key:a[:,rr,cc].T.copy() for key,a in arrays.items()}
    if family=="FUSION":
        X["FUSION"]=np.concatenate([X["RAW_ANNUAL"],
                                    X["S2_QUARTERLY"],X["EMB_ANNUAL"]],axis=1)
    if family=="FUSION_BRDF":
        X["FUSION_BRDF"]=np.concatenate([X["RAW_ANNUAL"],X["S2_BRDF_QUARTERLY"],X["EMB_ANNUAL"]],axis=1)
    if family=="FUSION_BRDF_T3Y":
        X["FUSION_BRDF_T3Y"]=np.concatenate([X["S2_BRDF_T3Y"],X["EMB_T3Y"]],axis=1)
    if family=="FUSION_AMAP":
        X["FUSION_AMAP"]=np.concatenate([X[k] for k in ("RAW_ANNUAL","S2_AMAP_QUARTERLY","EMB_ANNUAL")],axis=1)
    if family=="FUSION_AMAP_T3Y":
        X["FUSION_AMAP_T3Y"]=np.concatenate([X["S2_AMAP_T3Y"],X["EMB_T3Y"]],axis=1)
    if family=="FUSION_T3Y":
        X["FUSION_T3Y"]=np.concatenate([X["S2_T3Y"],X["EMB_T3Y"]],axis=1)
    y=target[:2,rr,cc].T.astype(np.float32)
    if not(np.isfinite(y).all() and y.min()>=0 and y.max()<=1.00001):
        raise RuntimeError(f"{site}: invalid reference targets")
    for family in X:
        if X[family].shape[1]!=len(FEATURES04[family]):
            raise RuntimeError(f"{site} {family}: feature count mismatch")
        if not np.isfinite(X[family]).all():
            raise RuntimeError(f"{site}/{family}: nonfinite feature")
    return dict(X=X,y=y,rows=rr,cols=cc,eligible=int(mask.sum()))
DATA07={}
COVERAGE07=[]
for family in ACTIVE06:
    DATA07[family]={}
    for i,site in enumerate(SITES):
        tr=load_site07(site,family,MAX_TRAIN07,SEED07+1000*i)
        te=load_site07(site,family,MAX_TEST07,SEED07+2000*i)
        DATA07[family][site]=dict(train=tr,test=te)
        COVERAGE07.append(dict(site=site,family=family,eligible=te["eligible"],
                               sampled=len(te["y"])))
    print(family,"reference cells:",{site:DATA07[family][site]["test"]["eligible"] for site in SITES})
pd.DataFrame(COVERAGE07).to_csv(OUTPUT07/"07_family_specific_support.csv",index=False)
print("Family-specific evaluation populations: raw RMSE values are NOT strictly matched between families.")
RECORDS07=[]
for family in ACTIVE06:
    for ti,target_name in enumerate(["musanga_probability_mean",
                                     "musanga_cover_fraction_p05"]):
        for heldout in SITES:
            Xtr=np.concatenate([DATA07[family][s]["train"]["X"][family]
                                for s in SITES if s!=heldout])
            ytr=np.concatenate([DATA07[family][s]["train"]["y"][:,ti]
                                for s in SITES if s!=heldout])
            test=DATA07[family][heldout]["test"]
            Xte=test["X"][family];truth=test["y"][:,ti]
            model=HistGradientBoostingRegressor(
                learning_rate=.06,max_iter=180,max_leaf_nodes=15,
                min_samples_leaf=60,l2_regularization=1.,
                loss="squared_error",random_state=SEED07)
            model.fit(Xtr,ytr)
            pred=np.clip(model.predict(Xte),0,1)
            baseline=np.full_like(truth,ytr.mean(),dtype=np.float32)
            for label,est in [("HGB",pred),("TRAIN_MEAN",baseline)]:
                RECORDS07.append(dict(family=family,target=target_name,
                     held_out_site=heldout,model=label,
                     n_train=len(ytr),n_test=len(truth),
                     observed_mean=float(truth.mean()),
                     predicted_mean=float(est.mean()),
                     mae=float(mean_absolute_error(truth,est)),
                     rmse=float(np.sqrt(mean_squared_error(truth,est))),
                     bias=float((est-truth).mean()),
                     r2=float(r2_score(truth,est)) if len(truth)>1 else np.nan))
            print(f"{family:13} | {target_name:27} | held out {heldout:9} "
                  f"| RMSE {RECORDS07[-2]['rmse']:.4f} "
                  f"| baseline {RECORDS07[-1]['rmse']:.4f}")
            if heldout=="Luki":
                pd.DataFrame(dict(row=test["rows"],col=test["cols"],
                                  observed=truth,predicted=pred,
                                  train_mean=baseline)).to_csv(
                    OUTPUT07/f"07_Luki_predictions_{family}_{target_name}.csv",
                    index=False)
METRICS07=pd.DataFrame(RECORDS07)
METRICS07.to_csv(OUTPUT07/"07_feature_family_LOSO_metrics_family_support.csv",index=False)
display(METRICS07[["family","target","held_out_site","model",
                    "n_train","n_test","observed_mean","predicted_mean",
                    "rmse","mae","bias","r2"]])
pivot07=METRICS07[(METRICS07.model=="HGB")&
    (METRICS07.target=="musanga_cover_fraction_p05")].pivot(
    index="held_out_site",columns="family",values="rmse")
print("Family-specific LOSO cover-fraction RMSE (different evaluation samples):")
display(pivot07.round(4))
print("Completed for families:",ACTIVE06)
print("This fixed HGB is a preliminary benchmark; nested tuning is reserved for later.")


## 08 — Multiscale Musanga aggregation and stand-to-landscape benchmarking (10–500 m)

**Run Blocks 00, 04, 05 and 06 first.** This is a **matched-scale exploratory benchmark**, using the same annual S2, quarterly S2, embeddings and fusion feature families when available. Resolution grid: **10, 30, 50, 100, 150, 250 and 500 m**, anchored to the identical native 10 m Sentinel-2 grid for each site. The 75 m experiment is deferred until deriving direct 75 m targets from the native UAV rasters, since it is not divisible by 10 m.

For scales above 10 m, this exploratory pass aggregates from the existing 10 m **90%-coverage-gated UAV references**. Cover/probability means are area-weighted by each cell's measured UAV coverage, and satellite predictors are averaged over matching joint-valid 10 m cells; cells with inadequate contributing area are excluded. This produces a conservative **valid 10 m reference-support fraction**, not the exact UAV area coverage of a coarse pixel. Particularly at 250–500 m, it can underestimate available high-resolution UAV coverage at boundaries. A final published multiscale study should aggregate **directly from the native full UAV probability maps** at each grid scale rather than stack 10 m thresholded values.

Uses a fixed ridge baseline at all scales to avoid comparing tree-model hyperparameters that become untenable with only a handful of 500 m cells. Five-site LOSO is attempted only where **all five sites have at least 10 supported coarse cells**, otherwise the scale is recorded as *insufficient* without pretending it is validated. Saves per-site support counts and per-fold metrics. **Note:** even 10–50 coarse samples per site can be highly spatially correlated; this is feasibility, not a final generalization estimate.

In [ ]:
# BLOCK 08 — Multiscale feasibility + matched-site RAW/embedding/fusion LOSO
# CPU only. Requires 00, 04, 05, 06. No new Earth Engine exports.
import numpy as np,pandas as pd,rasterio
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score
from IPython.display import display
SCALES08=[10,30,50,100,150,250,500]  # meters: all integer multiples of 10 m
MIN_SUPPORT08=0.90  # fraction of coarse cell with valid, aligned 10m reference+S2
MIN_SITE_CELLS08=10
MODEL08="StandardScaler+Ridge(alpha=10.0)"
OUTPUT08=OUT/"multiscale"
OUTPUT08.mkdir(parents=True,exist_ok=True)
assert "FEATURE_FILES06" in globals() and "ACTIVE06" in globals()
physical08=[f for f in ACTIVE06 if f not in ("FUSION","FUSION_T3Y","FUSION_BRDF","FUSION_BRDF_T3Y","FUSION_AMAP","FUSION_AMAP_T3Y")]
def coarse08(a,den,n):
    """Aggregate HxW array using a 10m support mask; denominator is fraction of 10m area."""
    h,w=a.shape
    hh=int(np.ceil(h/n));ww=int(np.ceil(w/n))
    out_h,out_w=hh*n,ww*n
    num=np.zeros((out_h,out_w),np.float64)
    d=np.zeros((out_h,out_w),np.float64)
    num[:h,:w]=a*den
    d[:h,:w]=den
    numerator=num.reshape(hh,n,ww,n).sum(axis=(1,3))
    denominator=d.reshape(hh,n,ww,n).sum(axis=(1,3))
    mean=np.full((hh,ww),np.nan,dtype=np.float32)
    np.divide(numerator,denominator,out=mean,where=denominator>0)
    return mean,(denominator/(n*n)).astype(np.float32)
def site08(site,scale,family):
    n=scale//10
    with rasterio.open(TARGET06[site]) as src:
        t=src.read([1,2,3]).astype(np.float32)
    valid=(np.isfinite(t).all(axis=0)&(t[0]!=-9999)&(t[1]!=-9999)&
           (t[2]>=.90-1e-5))
    arrays={}
    required={"FUSION":["RAW_ANNUAL","S2_QUARTERLY","EMB_ANNUAL"],"FUSION_T3Y":["S2_T3Y","EMB_T3Y"],"FUSION_BRDF":["RAW_ANNUAL","S2_BRDF_QUARTERLY","EMB_ANNUAL"],"FUSION_BRDF_T3Y":["S2_BRDF_T3Y","EMB_T3Y"],"FUSION_AMAP":["RAW_ANNUAL","S2_AMAP_QUARTERLY","EMB_ANNUAL"],"FUSION_AMAP_T3Y":["S2_AMAP_T3Y","EMB_T3Y"]}.get(family,[family])
    for part in required:
        with rasterio.open(FEATURE_FILES06[part][site]) as src:
            data=src.read().astype(np.float32)
        good=np.isfinite(data).all(axis=0)&np.all(data!=-9999,axis=0)
        if part=="RAW_ANNUAL":good &= data[14]>0
        if part in ("S2_QUARTERLY","S2_BRDF_QUARTERLY","S2_AMAP_QUARTERLY"):good &= np.all(data[[10,21,32,43]]>0,axis=0)
        valid &= good
        arrays[part]=data
    # Valid 10m reference cells for THIS requested family only.
    weights=np.where(valid,np.clip(t[2],0,1),0).astype(np.float32)
    ref,coverage=coarse08(np.where(valid,t[1],0),weights,n)
    probability,_=coarse08(np.where(valid,t[0],0),weights,n)
    keep=(coverage>=MIN_SUPPORT08-1e-6)&np.isfinite(ref)&np.isfinite(probability)
    feat={}
    for part,arr in arrays.items():
        bands=[coarse08(np.where(valid,arr[i],0),weights,n)[0] for i in range(arr.shape[0])]
        feat[part]=np.stack(bands,axis=-1)
        keep &= np.isfinite(feat[part]).all(axis=-1)
    coords=np.column_stack(np.where(keep))
    X={family:val[keep] for family,val in feat.items()}
    if family=="FUSION":
        X["FUSION"]=np.concatenate([X[k] for k in ("RAW_ANNUAL","S2_QUARTERLY","EMB_ANNUAL")],axis=1)
    if family=="FUSION_BRDF":
        X["FUSION_BRDF"]=np.concatenate([X[k] for k in ("RAW_ANNUAL","S2_BRDF_QUARTERLY","EMB_ANNUAL")],axis=1)
    if family=="FUSION_BRDF_T3Y":
        X["FUSION_BRDF_T3Y"]=np.concatenate([X["S2_BRDF_T3Y"],X["EMB_T3Y"]],axis=1)
    if family=="FUSION_AMAP":
        X["FUSION_AMAP"]=np.concatenate([X[k] for k in ("RAW_ANNUAL","S2_AMAP_QUARTERLY","EMB_ANNUAL")],axis=1)
    if family=="FUSION_AMAP_T3Y":
        X["FUSION_AMAP_T3Y"]=np.concatenate([X["S2_AMAP_T3Y"],X["EMB_T3Y"]],axis=1)
    if family=="FUSION_T3Y":
        X["FUSION_T3Y"]=np.concatenate([X["S2_T3Y"],X["EMB_T3Y"]],axis=1)
    Y=np.column_stack([probability[keep],ref[keep]])
    return dict(X=X,Y=Y,coords=coords,coverage=coverage,
                n_cells=len(coords),n_grid_cells=coverage.size,
                median_supported_coverage=float(np.median(coverage[keep])) if keep.any() else np.nan)
support08=[];metrics08=[]
for scale in SCALES08:
    for family in ACTIVE06:
        ds={}
        for site in SITES:
            ds[site]=site08(site,scale,family)
            support08.append(dict(scale_m=scale,site=site,family=family,
                usable_cells=ds[site]["n_cells"],total_grid_cells=ds[site]["n_grid_cells"],
                median_usable_coverage=ds[site]["median_supported_coverage"],
                coverage_cutoff=MIN_SUPPORT08))
        counts={site:ds[site]["n_cells"] for site in SITES}
        print(f"{scale:>3}m {family:13} supported cells:",counts)
        # Missing support in Luki must not prevent evaluation of other held-out sites.
        for ti,target_name in enumerate(("musanga_probability_mean","musanga_cover_fraction_p05")):
            for test_site in SITES:
                train_sites=[site for site in SITES if site!=test_site and ds[site]["n_cells"]>=MIN_SITE_CELLS08]
                if len(train_sites)<2 or ds[test_site]["n_cells"]<MIN_SITE_CELLS08:
                    continue
                Xtr=np.concatenate([ds[site]["X"][family] for site in train_sites])
                ytr=np.concatenate([ds[site]["Y"][:,ti] for site in train_sites])
                Xte=ds[test_site]["X"][family]
                truth=ds[test_site]["Y"][:,ti]
                model=make_pipeline(StandardScaler(),Ridge(alpha=10.0))
                model.fit(Xtr,ytr)
                pred=np.clip(model.predict(Xte),0,1)
                baseline=np.full_like(truth,ytr.mean())
                for label,estimate in (("RIDGE",pred),("TRAIN_MEAN",baseline)):
                    metrics08.append(dict(scale_m=scale,family=family,target=target_name,
                        held_out_site=test_site,model=label,n_train=len(ytr),
                        n_test=len(truth),training_sites="+".join(train_sites),
                        truth_mean=float(truth.mean()),pred_mean=float(estimate.mean()),
                        mae=float(mean_absolute_error(truth,estimate)),
                        rmse=float(np.sqrt(mean_squared_error(truth,estimate))),
                        bias=float((estimate-truth).mean()),
                        r2=float(r2_score(truth,estimate)) if len(truth)>1 else np.nan))
qa08=pd.DataFrame(support08)
qa08.to_csv(OUTPUT08/"08_multiscale_site_support_by_family.csv",index=False)
display(qa08.pivot_table(index=["site","family"],columns="scale_m",values="usable_cells"))
if metrics08:
    score08=pd.DataFrame(metrics08)
    score08.to_csv(OUTPUT08/"08_multiscale_LOSO_metrics_by_family.csv",index=False)
    main08=score08[(score08.target=="musanga_cover_fraction_p05")&
                   (score08.model=="RIDGE")]
    display(main08.pivot_table(index=["scale_m","held_out_site"],
                columns="family",values="rmse").round(4))
else:
    print("No held-out fold met the minimum; inspect per-family support.")
print("All outputs saved:",OUTPUT08)
print("CAUTION: At coarse scales, training folds may include fewer than four sites.")
print("Compare models using the same training sites and test support; family-specific RMSE is not always matched.")
print("Approximate coarse targets: aggregate of validated 10 m UAV references,")
print("not a replacement for direct full-resolution UAV aggregation at 250–500m.")


## 09 — Diagnostics, limitations and planned BRDF option

The former separate Luki spectral-domain-shift diagnostic and earlier RAW benchmark remain available in the [historical notebook at the October 2026 GitHub commit](https://github.com/kvanoost/Musanga/blob/1b107b864d018eb489dd5b21ed5c97d9cd11b3bb/KVO_08_Satellite_Musanga_Fraction.ipynb). In this revision, Section 07 saves predictions and errors for Luki **for every feature family**, so domain-shift plots and ecological interpretation can be generated from one harmonized result.

**Future branch — Ploton/PP tropical BRDF normalization.** The user-supplied Earth Engine source from July 2026 includes RTLS/Li kernels, a tile-orbit asset `users/wiell/SepalResources/sentinel2TileOrbits` and nine-band AMAP coefficients. Run correction on each input S2 acquisition (over buffered study areas), **before** compositing, then compare on the same pixels and years as RAW. Google `COPERNICUS/S2_SR_HARMONIZED` has already harmonized the baseline radiometric offset: do not subtract 1,000 DN again. The published AMAP table has **no B8A coefficients**. The previous safe BRDF preflight remains in the [archived NB08 revision](https://github.com/kvanoost/Musanga/blob/706a5063fba94fa766e437658575093e5309a48e/KVO_08_Satellite_Musanga_Fraction.ipynb) and can be integrated when RAW feature-family comparison is complete.

**Next scientific tests after this benchmark:** quarterly phenological context, embedding-only vs S2-only vs fusion generalization, 10/30/50/75/100m target aggregation, nested LOSO model selection, calibration checks on U-Net fractional proxies, and spatial-block uncertainty. Do not treat the annual satellite embedding native 10m *pixel spacing* as evidence for independent 10m observational support.